# Pengantar Regex untuk NLP

Notebook ini disusun sebagai bahan ajar khusus untuk memahami **regular expression (regex)** dari dasar sampai penerapan praktis, terutama untuk **tokenization** dan **text preprocessing** dalam NLP.

Cakupan:
- definisi regex
- mengapa regex penting dalam NLP
- raw string di Python
- karakter literal dan metacharacter
- character class
- quantifier
- anchors
- grouping dan alternation
- escaping
- fungsi-fungsi penting pada modul `re`
- contoh-contoh regex untuk preprocessing
- regex tokenizer
- kelebihan dan keterbatasan regex
- latihan reflektif

## Tujuan Pembelajaran

Setelah mempelajari notebook ini, mahasiswa diharapkan mampu:

1. menjelaskan apa itu regex dan kegunaannya
2. membaca pola regex dasar dengan benar
3. menggunakan `re.findall`, `re.search`, `re.match`, `re.sub`, dan `re.split`
4. membuat pola regex sederhana untuk kebutuhan preprocessing
5. memahami bagaimana regex digunakan untuk tokenization dalam NLP
6. mengenali keterbatasan pendekatan rule-based berbasis regex

In [5]:

import re
from typing import List

## 1. Apa itu Regex?

**Regex** adalah singkatan dari **regular expression**, yaitu pola teks yang digunakan untuk:

- mencari bagian tertentu dari string
- mencocokkan pola
- mengambil potongan teks
- memisahkan string
- mengganti bagian teks

Secara sederhana, regex adalah **bahasa mini untuk mendeskripsikan pola teks**.

### Contoh ide sederhananya
- mencari semua angka dalam kalimat
- mengambil semua email dalam dokumen
- mendeteksi URL
- mengambil hashtag dari tweet
- memisahkan kata dari tanda baca

## 2. Mengapa Regex Penting dalam NLP?

Dalam NLP, kita sering berhadapan dengan teks mentah yang berisi:

- huruf besar dan kecil
- angka
- tanda baca
- URL
- mention
- hashtag
- simbol
- format campuran seperti `COVID-19`, `Rp10.000`, `code=500`

Kalau kita hanya memakai `.split()`, pemisahan teks hanya berdasarkan spasi.  
Akibatnya, token seperti `pembangunan,` atau `NLP!` masih membawa simbol.

Regex memberi kita cara yang lebih presisi untuk mendefinisikan:
- apa yang dianggap kata
- apa yang dianggap angka
- apa yang ingin dihapus
- apa yang ingin dipertahankan

In [6]:
text = "Pemerintah sedang melakukan pembangunan, pengembangan, dan evaluasi NLP di Indonesia."

print("split() biasa:")
print(text.split())

print("\nregex sederhana [A-Za-z]+:")
print(re.findall(r"[A-Za-z]+", text))

split() biasa:
['Pemerintah', 'sedang', 'melakukan', 'pembangunan,', 'pengembangan,', 'dan', 'evaluasi', 'NLP', 'di', 'Indonesia.']

regex sederhana [A-Za-z]+:
['Pemerintah', 'sedang', 'melakukan', 'pembangunan', 'pengembangan', 'dan', 'evaluasi', 'NLP', 'di', 'Indonesia']


## 3. Raw String di Python

Regex di Python hampir selalu ditulis dengan awalan `r`, misalnya:

```python
r"\d+"
```

Ini disebut **raw string**.

Tujuannya agar backslash `\` tidak diinterpretasikan dulu oleh Python.

### Contoh
- `r"\d+"` lebih nyaman daripada `"\\d+"`
- `r"\s+"` lebih mudah dibaca daripada `"\\s+"`

Dalam praktik, hampir selalu gunakan **raw string** saat menulis regex.

In [7]:

patterns = [
    r"\d+",
    "\\d+"
]

for p in patterns:
    print(p)

\d+
\d+


## 4. Konsep Dasar Regex

Regex dibangun dari kombinasi:
- karakter literal
- metacharacter
- character class
- quantifier
- grouping
- anchors

Kita bahas satu per satu.

### 4.1 Karakter Literal

Karakter literal berarti karakter dicocokkan apa adanya.

Contoh:
- `data` cocok dengan string `"data"`
- `NLP` cocok dengan `"NLP"`

In [8]:

text = "Saya belajar data science dan NLP"

print(re.findall(r"data", text))
print(re.findall(r"NLP", text))

['data']
['NLP']


### 4.2 Character Class: `[]`

Character class menyatakan himpunan karakter yang boleh cocok.

Contoh:
- `[abc]` → satu karakter: `a`, `b`, atau `c`
- `[A-Z]` → satu huruf kapital
- `[a-z]` → satu huruf kecil
- `[0-9]` → satu digit

In [9]:

text = "A1b2C3d4"

print("Huruf kecil:", re.findall(r"[a-z]", text))
print("Huruf besar:", re.findall(r"[A-Z]", text))
print("Digit      :", re.findall(r"[0-9]", text))

Huruf kecil: ['b', 'd']
Huruf besar: ['A', 'C']
Digit      : ['1', '2', '3', '4']


### 4.3 Range

#### Apa itu Range?

**Range** adalah cara singkat untuk menyatakan **rentang karakter** di dalam **character class** `[]`.

Contoh:

```python
[A-Z]
```

artinya:

> cocok dengan **satu karakter** yang berada di antara `A` sampai `Z`.

Dengan kata lain, `[A-Z]` adalah bentuk singkat dari semua huruf kapital dari `A` sampai `Z`.



#### Range hanya berlaku di dalam `[]`

Ini penting. Bentuk seperti:

```python
A-Z
```

baru bermakna sebagai **range** jika ditulis di dalam **character class**, yaitu:

```python
[A-Z]
```

Kalau ditulis di luar `[]`, maka `-` hanya dianggap karakter biasa.

#### Contoh dasar range

##### 1. Huruf kapital
```python
[A-Z]
```
Artinya: satu huruf kapital dari A sampai Z.

##### 2. Huruf kecil
```python
[a-z]
```
Artinya: satu huruf kecil dari a sampai z.

##### 3. Digit
```python
[0-9]
```
Artinya: satu digit dari 0 sampai 9.

#### Range dapat digabung

Kita bisa menggabungkan beberapa range dalam satu character class.

Contoh:

```python
[A-Za-z]
```

Artinya:
- satu huruf kapital `A-Z`
- atau satu huruf kecil `a-z`

Jadi pola ini berarti: **satu huruf alfabet Latin**.

Contoh lain:

```python
[A-Za-z0-9]
```

Artinya:
- satu huruf kapital
- atau satu huruf kecil
- atau satu digit

Jadi pola ini berarti: **satu karakter alfanumerik**.

#### Range hanya berarti satu karakter

Ini bagian yang sangat penting.

Pola:

```python
[a-z]
```

**hanya cocok dengan satu karakter huruf kecil**, bukan satu kata.

Contoh:

```python
re.findall(r"[a-z]", "data")
```

hasilnya:

```python
['d', 'a', 't', 'a']
```

Kalau ingin mencocokkan satu kata penuh, kita perlu menambahkan **quantifier** seperti `+`:

```python
[a-z]+
```

yang berarti:

> satu atau lebih huruf kecil secara berurutan

In [ ]:
print(re.findall(r"[a-z]", "data"))
print(re.findall(r"[a-z]+", "data"))

['d', 'a', 't', 'a']
['data']


In [12]:
text = "Data2026NLP"
print(re.findall(r"[A-Za-z]+", text))
print(re.findall(r"[A-Za-z0-9]+", text))

['Data', 'NLP']
['Data2026NLP']


#### Perbedaan penting

- `[A-Z]` → satu huruf kapital
- `[A-Z]+` → satu atau lebih huruf kapital berurutan
- `[0-9]` → satu digit
- `[0-9]+` → satu atau lebih digit
- `[A-Za-z0-9]+` → satu atau lebih huruf/angka berurutan

In [13]:

samples = [
    "data",
    "NLP",
    "2026",
    "Data2026",
    "NLP_2026",
]

patterns = [
    r"[a-z]",
    r"[a-z]+",
    r"[A-Z]",
    r"[A-Z]+",
    r"[0-9]",
    r"[0-9]+",
    r"[A-Za-z]",
    r"[A-Za-z]+",
    r"[A-Za-z0-9]+",
    r"[A-Za-z0-9_]+",
]

print("DEMO RANGE DALAM REGEX")
print("=" * 80)

for text in samples:
    print(f"\nTEXT: {text}")
    for p in patterns:
        print(f"{p:20} -> {re.findall(p, text)}")

DEMO RANGE DALAM REGEX

TEXT: data
[a-z]                -> ['d', 'a', 't', 'a']
[a-z]+               -> ['data']
[A-Z]                -> []
[A-Z]+               -> []
[0-9]                -> []
[0-9]+               -> []
[A-Za-z]             -> ['d', 'a', 't', 'a']
[A-Za-z]+            -> ['data']
[A-Za-z0-9]+         -> ['data']
[A-Za-z0-9_]+        -> ['data']

TEXT: NLP
[a-z]                -> []
[a-z]+               -> []
[A-Z]                -> ['N', 'L', 'P']
[A-Z]+               -> ['NLP']
[0-9]                -> []
[0-9]+               -> []
[A-Za-z]             -> ['N', 'L', 'P']
[A-Za-z]+            -> ['NLP']
[A-Za-z0-9]+         -> ['NLP']
[A-Za-z0-9_]+        -> ['NLP']

TEXT: 2026
[a-z]                -> []
[a-z]+               -> []
[A-Z]                -> []
[A-Z]+               -> []
[0-9]                -> ['2', '0', '2', '6']
[0-9]+               -> ['2026']
[A-Za-z]             -> []
[A-Za-z]+            -> []
[A-Za-z0-9]+         -> ['2026']
[A-Za-z0-9_]+        ->

#### Contoh NLP yang relevan

Teks:

```text
Pemerintah 2026
```

Regex:

```python
[A-Za-z]+
```

Hasil:
- `Pemerintah`

Regex:

```python
[0-9]+
```

Hasil:
- `2026`

Regex:

```python
[A-Za-z0-9]+
```

Hasil:
- `Pemerintah`
- `2026`

In [14]:
texts = [
    "Pemerintah 2026",
    "NLP_2026",
    "COVID-19",
    "data science",
]

print("CONTOH NLP TERKAIT RANGE")
print("=" * 80)

for text in texts:
    print(f"\nInput: {text}")
    print("[A-Za-z]+      ->", re.findall(r"[A-Za-z]+", text))
    print("[0-9]+         ->", re.findall(r"[0-9]+", text))
    print("[A-Za-z0-9]+   ->", re.findall(r"[A-Za-z0-9]+", text))
    print("[A-Za-z0-9_]+  ->", re.findall(r"[A-Za-z0-9_]+", text))

CONTOH NLP TERKAIT RANGE

Input: Pemerintah 2026
[A-Za-z]+      -> ['Pemerintah']
[0-9]+         -> ['2026']
[A-Za-z0-9]+   -> ['Pemerintah', '2026']
[A-Za-z0-9_]+  -> ['Pemerintah', '2026']

Input: NLP_2026
[A-Za-z]+      -> ['NLP']
[0-9]+         -> ['2026']
[A-Za-z0-9]+   -> ['NLP', '2026']
[A-Za-z0-9_]+  -> ['NLP_2026']

Input: COVID-19
[A-Za-z]+      -> ['COVID']
[0-9]+         -> ['19']
[A-Za-z0-9]+   -> ['COVID', '19']
[A-Za-z0-9_]+  -> ['COVID', '19']

Input: data science
[A-Za-z]+      -> ['data', 'science']
[0-9]+         -> []
[A-Za-z0-9]+   -> ['data', 'science']
[A-Za-z0-9_]+  -> ['data', 'science']


#### Tentang underscore

Teks:

```text
NLP_2026
```

Regex:

```python
[A-Za-z0-9]+
```

Hasil:
- `NLP`
- `2026`

karena underscore `_` **tidak termasuk**.

Kalau ingin underscore ikut dianggap bagian token, gunakan:

```python
[A-Za-z0-9_]+
```

Hasil:
- `NLP_2026`

#### Tentang tanda minus `-`

Di dalam `[]`, tanda minus `-` biasanya menandakan **range**.

Contoh:

```python
[a-z]
```

Tetapi jika kita ingin mencocokkan minus literal, biasanya `-` diletakkan di awal atau akhir character class, misalnya:

```python
[-A-Za-z0-9]
```

atau

```python
[A-Za-z0-9-]
```

In [15]:

minus_text = "COVID-19 anak-anak e-commerce"

print("Minus sebagai literal di akhir class:")
print(re.findall(r"[A-Za-z0-9-]+", minus_text))

print("\nTanpa minus di class:")
print(re.findall(r"[A-Za-z0-9]+", minus_text))

Minus sebagai literal di akhir class:
['COVID-19', 'anak-anak', 'e-commerce']

Tanpa minus di class:
['COVID', '19', 'anak', 'anak', 'e', 'commerce']


#### Range `À-ÿ`

Pada pola seperti:

```python
[A-Za-zÀ-ÿ0-9_]+
```

bagian `À-ÿ` adalah range untuk beberapa karakter Latin beraksen, misalnya:
- `é`
- `à`
- `ü`
- `ñ`

Tujuannya untuk sedikit memperluas cakupan huruf Latin.  
Namun ini tetap bukan solusi universal untuk semua bahasa di dunia.

In [16]:

accent_text = "café déjà niño über año"

print(re.findall(r"[A-Za-z]+", accent_text))
print(re.findall(r"[A-Za-zÀ-ÿ]+", accent_text))

['caf', 'd', 'j', 'ni', 'o', 'ber', 'a', 'o']
['café', 'déjà', 'niño', 'über', 'año']


#### Intuisi sederhana

Bayangkan `[]` seperti sebuah **kotak pilihan karakter**.

- `[a-z]` berarti: pilih **satu karakter** dari semua huruf kecil a sampai z
- `[a-z]+` berarti: ambil **rangkaian karakter** selama semuanya masih huruf kecil

#### Kesalahan Umum 

1. Mengira `[a-z]` berarti satu kata kecil  
   Padahal itu hanya **satu karakter kecil**.

2. Mengira `[A-Za-z]` berarti satu kata  
   Padahal itu hanya **satu huruf kapital atau kecil**.

3. Lupa bahwa untuk mencocokkan kata penuh, perlu quantifier seperti `+`.

#### Ringkasan

- Range adalah penulisan singkat untuk sekumpulan karakter berurutan
- Range hanya berlaku di dalam `[]`
- `[a-z]` berarti satu huruf kecil
- `[0-9]` berarti satu digit
- Untuk mencocokkan banyak karakter, gunakan quantifier, misalnya:
  - `[a-z]+`
  - `[0-9]+`
  - `[A-Za-z0-9]+`

### 4.4 Quantifier

#### Apa itu Quantifier?

**Quantifier** adalah simbol dalam regex yang menyatakan **berapa kali** suatu pola boleh muncul.

Kalau **range** atau **character class** menjelaskan *karakter apa yang boleh cocok*, maka **quantifier** menjelaskan *berapa banyak kemunculan pola itu*.

Contoh sederhana:

```python
[a-z]
```

berarti:
> satu huruf kecil

Tetapi:

```python
[a-z]+
```

berarti:
> satu atau lebih huruf kecil secara berurutan

Jadi, quantifier mengubah pola dari **satu unit** menjadi **sekumpulan unit**.

#### Mengapa Quantifier penting?

Dalam NLP dan text preprocessing, kita sering tidak hanya ingin mencocokkan satu karakter, tetapi:
- satu kata penuh
- satu angka penuh
- beberapa spasi beruntun
- pola yang boleh ada atau tidak ada
- pola dengan jumlah kemunculan tertentu

Tanpa quantifier, regex sering hanya menangkap satu karakter per satu waktu.

#### Quantifier utama yang wajib dipahami

Quantifier paling dasar dan paling sering dipakai adalah:

- `+`  → satu kali atau lebih
- `*`  → nol kali atau lebih
- `?`  → nol atau satu kali
- `{n}` → tepat n kali
- `{m,n}` → antara m sampai n kali
- `{m,}` → minimal m kali

#### 1. Quantifier `+`

Pola:

```python
[a-z]+
```

artinya:

> satu atau lebih huruf kecil berurutan

### Intuisi
Kalau `[a-z]` adalah “satu huruf kecil”, maka `[a-z]+` adalah “sekumpulan huruf kecil yang berurutan”.

### Contoh
- `data` → cocok sebagai satu unit
- `abc` → cocok sebagai satu unit
- `123` → tidak cocok

In [17]:

print(re.findall(r"[a-z]", "data"))
print(re.findall(r"[a-z]+", "data"))

['d', 'a', 't', 'a']
['data']


##### Penjelasan
- `[a-z]` menghasilkan `d`, `a`, `t`, `a` satu per satu
- `[a-z]+` menghasilkan `data` sebagai satu rangkaian

Jadi `+` sangat penting untuk mengubah pola dari “karakter” menjadi “token”.

In [18]:

samples = ["data", "abc123", "NLP", "machinelearning"]

print("DEMO QUANTIFIER +")
print("=" * 80)

for text in samples:
    print(f"\nTEXT: {text}")
    print("[a-z]   ->", re.findall(r"[a-z]", text))
    print("[a-z]+  ->", re.findall(r"[a-z]+", text))
    print("[A-Z]+  ->", re.findall(r"[A-Z]+", text))
    print("[0-9]+  ->", re.findall(r"[0-9]+", text))

DEMO QUANTIFIER +

TEXT: data
[a-z]   -> ['d', 'a', 't', 'a']
[a-z]+  -> ['data']
[A-Z]+  -> []
[0-9]+  -> []

TEXT: abc123
[a-z]   -> ['a', 'b', 'c']
[a-z]+  -> ['abc']
[A-Z]+  -> []
[0-9]+  -> ['123']

TEXT: NLP
[a-z]   -> []
[a-z]+  -> []
[A-Z]+  -> ['NLP']
[0-9]+  -> []

TEXT: machinelearning
[a-z]   -> ['m', 'a', 'c', 'h', 'i', 'n', 'e', 'l', 'e', 'a', 'r', 'n', 'i', 'n', 'g']
[a-z]+  -> ['machinelearning']
[A-Z]+  -> []
[0-9]+  -> []


#### 2. Quantifier `*`

Pola:

```python
[a-z]*
```

artinya:

> nol kali atau lebih huruf kecil

Ini terdengar mirip dengan `+`, tetapi ada perbedaan besar:

- `+` mensyaratkan **minimal satu**
- `*` mengizinkan **nol**

##### Implikasi
`*` dapat mencocokkan string kosong.

In [19]:

texts = ["data", "123", ""]

print("DEMO QUANTIFIER *")
print("=" * 80)

for t in texts:
    print(f"Input: {repr(t)}")
    print(re.findall(r"[a-z]*", t))
    print("-" * 60)

DEMO QUANTIFIER *
Input: 'data'
['data', '']
------------------------------------------------------------
Input: '123'
['', '', '', '']
------------------------------------------------------------
Input: ''
['']
------------------------------------------------------------


##### Catatan penting
Karena `*` mengizinkan kecocokan kosong, hasil `findall()` sering tampak “aneh” bagi pemula.  
Itulah sebabnya `*` harus dipakai dengan hati-hati, terutama saat belajar awal.

#### 3. Quantifier `?`

Pola:

```python
colou?r
```

artinya:

> huruf `u` boleh ada atau tidak ada

Jadi pola itu cocok dengan:
- `color`
- `colour`

### Makna umum
`?` berarti:
> nol atau satu kali

Ini sangat berguna untuk membuat bagian pola yang bersifat opsional.

In [20]:

text = "color colour colouur"

print(re.findall(r"colou?r", text))

['color', 'colour']


#### Contoh NLP
Quantifier `?` sering dipakai ketika sebuah pola memiliki variasi kecil, misalnya:
- ejaan alternatif
- bagian yang opsional
- awalan/akhiran yang boleh ada atau tidak

In [21]:

samples = [
    "organization organisation",
    "color colour",
    "analyze analyse"
]

patterns = [
    r"organi[sz]ation",
    r"colou?r",
    r"analy[sz]e"
]

print("DEMO QUANTIFIER ? DAN VARIASI EJAAN")
print("=" * 80)

for text in samples:
    print("\nTEXT:", text)
    for p in patterns:
        print(f"{p:20} -> {re.findall(p, text)}")

DEMO QUANTIFIER ? DAN VARIASI EJAAN

TEXT: organization organisation
organi[sz]ation      -> ['organization', 'organisation']
colou?r              -> []
analy[sz]e           -> []

TEXT: color colour
organi[sz]ation      -> []
colou?r              -> ['color', 'colour']
analy[sz]e           -> []

TEXT: analyze analyse
organi[sz]ation      -> []
colou?r              -> []
analy[sz]e           -> ['analyze', 'analyse']


#### 4. Quantifier Tepat `{n}`

Pola:

```python
\d{4}
```

artinya:

> tepat 4 digit

Contoh cocok:
- `2026`
- `1999`

Contoh tidak cocok penuh:
- `26`
- `123`

In [22]:

text = "Tahun 2026, 1999, dan 88"

print(re.findall(r"\d{4}", text))

['2026', '1999']


##### Kegunaan
Quantifier `{n}` cocok untuk pola yang jumlah karakternya tetap, misalnya:
- tahun 4 digit
- kode OTP 6 digit
- format ID tertentu

#### 5. Quantifier Rentang `{m,n}`

Pola:

```python
\d{2,4}
```

artinya:

> cocok dengan angka yang panjangnya antara 2 sampai 4 digit

Contoh cocok:
- `12`
- `202`
- `2026`

Contoh tidak cocok:
- `1`
- `12345`

In [23]:

text = "Angka: 1 12 202 2026 12345"

print(re.findall(r"\d{2,4}", text))

['12', '202', '2026', '1234']


#### 6. Quantifier Minimal `{m,}`

Pola:

```python
[a-z]{3,}
```

artinya:

> tiga huruf kecil atau lebih

Contoh cocok:
- `data`
- `ilmu`
- `regex`

Contoh tidak cocok:
- `di`
- `ke`

In [24]:

text = "di ke data ilmu regex ai"

print(re.findall(r"[a-z]{3,}", text))

['data', 'ilmu', 'regex']


#### 7. Perbandingan Quantifier

Mari lihat ringkasannya:

- `[a-z]`   → tepat satu huruf kecil
- `[a-z]+`  → satu atau lebih huruf kecil
- `[a-z]*`  → nol atau lebih huruf kecil
- `[a-z]?`  → nol atau satu huruf kecil
- `[a-z]{3}`   → tepat tiga huruf kecil
- `[a-z]{2,4}` → dua sampai empat huruf kecil
- `[a-z]{3,}`  → minimal tiga huruf kecil

In [25]:

patterns = [
    r"[a-z]",
    r"[a-z]+",
    r"[a-z]*",
    r"[a-z]?",
    r"[a-z]{3}",
    r"[a-z]{2,4}",
    r"[a-z]{3,}",
]

text = "data"

print("PERBANDINGAN QUANTIFIER")
print("=" * 80)
print("TEXT:", text)

for p in patterns:
    print(f"{p:15} -> {re.findall(p, text)}")

PERBANDINGAN QUANTIFIER
TEXT: data
[a-z]           -> ['d', 'a', 't', 'a']
[a-z]+          -> ['data']
[a-z]*          -> ['data', '']
[a-z]?          -> ['d', 'a', 't', 'a', '']
[a-z]{3}        -> ['dat']
[a-z]{2,4}      -> ['data']
[a-z]{3,}       -> ['data']


#### 8. Quantifier pada Pola Angka

Quantifier sangat sering dipakai untuk angka.

Contoh:
- `\d+` → satu atau lebih digit
- `\d{4}` → tepat 4 digit
- `\d{2,4}` → 2 sampai 4 digit

In [26]:

text = "Nomor: 7 25 202 2026 12345"

print("\d+      ->", re.findall(r"\d+", text))
print("\d{4}    ->", re.findall(r"\d{4}", text))
print("\d{2,4}  ->", re.findall(r"\d{2,4}", text))
print("\d{3,}   ->", re.findall(r"\d{3,}", text))

\d+      -> ['7', '25', '202', '2026', '12345']
\d{4}    -> ['2026', '1234']
\d{2,4}  -> ['25', '202', '2026', '1234']
\d{3,}   -> ['202', '2026', '12345']


#### 9. Quantifier dan Spasi

Dalam preprocessing, quantifier sering dipakai untuk merapikan spasi.

Pola:

```python
\s+
```

artinya:

> satu atau lebih whitespace

Ini sangat umum dipakai untuk menormalkan spasi berlebih menjadi satu spasi.

In [27]:

text = "Ini    spasi\tberlebih\nsekali"

print("Asli :", repr(text))
print("Hasil:", re.sub(r"\s+", " ", text).strip())

Asli : 'Ini    spasi\tberlebih\nsekali'
Hasil: Ini spasi berlebih sekali


#### 10. Quantifier dan Tokenization

Regex tokenizer hampir selalu bergantung pada quantifier, karena kita ingin menangkap rangkaian karakter, bukan karakter tunggal.

Contoh:

```python
[A-Za-z]+
```

berarti:
> satu atau lebih huruf

Kalau tanpa `+`, hasilnya akan pecah menjadi karakter satu-satu.

In [28]:

def regex_word_tokenizer_basic(text: str):
    return re.findall(r"[A-Za-z]+", text)

def regex_word_tokenizer_char_level(text: str):
    return re.findall(r"[A-Za-z]", text)

sample = "Pemerintah belajar NLP"

print("Tanpa quantifier + :", regex_word_tokenizer_char_level(sample))
print("Dengan quantifier +:", regex_word_tokenizer_basic(sample))

Tanpa quantifier + : ['P', 'e', 'm', 'e', 'r', 'i', 'n', 't', 'a', 'h', 'b', 'e', 'l', 'a', 'j', 'a', 'r', 'N', 'L', 'P']
Dengan quantifier +: ['Pemerintah', 'belajar', 'NLP']


#### 11. Quantifier dan Bagian Opsional

Misalnya kita ingin mencocokkan bentuk:
- `don't`
- `cant`
- `it's`

Kita bisa membuat bagian apostrof menjadi opsional.

In [29]:

pattern = r"[A-Za-z]+(?:'[A-Za-z]+)?"
text = "I don't know if its or it's correct"

print(re.findall(pattern, text))

['I', "don't", 'know', 'if', 'its', 'or', "it's", 'correct']


##### Analisis
Pada pola:

```python
[A-Za-z]+(?:'[A-Za-z]+)?
```

- `[A-Za-z]+` menangkap kata utama
- `(?:'[A-Za-z]+)?` menangkap bagian apostrof yang opsional

Jadi quantifier `?` di sini membuat bagian apostrof boleh ada atau tidak.

#### 12. Quantifier dan Greedy Behavior

Secara default, banyak quantifier bersifat **greedy**, artinya mereka mencoba mencocokkan sebanyak mungkin karakter.

Contoh:
- `.*` akan mengambil sebanyak mungkin
- `.+` juga akan mengambil sebanyak mungkin selama masih cocok

In [30]:

text = "<title>Regex</title><title>NLP</title>"

print("Greedy  :", re.findall(r"<title>.*</title>", text))
print("Non-greedy:", re.findall(r"<title>.*?</title>", text))

Greedy  : ['<title>Regex</title><title>NLP</title>']
Non-greedy: ['<title>Regex</title>', '<title>NLP</title>']


##### Penjelasan
- `.*` → nol atau lebih karakter, sebanyak mungkin
- `.*?` → nol atau lebih karakter, tetapi seminimal mungkin

Walaupun ini topik lanjutan, penting diperkenalkan agar mahasiswa tahu quantifier juga punya perilaku pencocokan.

#### 13. Kelebihan dan Risiko Tiap Quantifier

##### `+`
- sangat umum
- bagus untuk tokenization
- aman untuk “satu atau lebih”

##### `*`
- fleksibel
- tetapi sering menghasilkan match kosong
- harus hati-hati saat dipakai dengan `findall`

##### `?`
- bagus untuk pola opsional
- berguna untuk variasi bentuk

##### `{n}`, `{m,n}`, `{m,}`
- bagus untuk format yang terstruktur
- cocok untuk angka, kode, ID, tahun, OTP

#### 14. Kesalahan Umum 

1. Mengira `[a-z]` dan `[a-z]+` sama  
   Padahal yang pertama satu karakter, yang kedua satu atau lebih karakter.

2. Tidak sadar bahwa `*` bisa mencocokkan string kosong.

3. Mengira `?` berarti “apa saja”  
   Padahal `?` berarti nol atau satu kali.

4. Salah memahami `{2,4}` sebagai “harus 2 dan 4”  
   Padahal artinya “antara 2 sampai 4 kali”.

5. Tidak memahami bahwa quantifier default bersifat greedy.

In [31]:

texts = ["a", "ab", "", "2026", "12", "12345"]

patterns = [
    r"[a-z]",
    r"[a-z]+",
    r"[a-z]*",
    r"[a-z]?",
    r"\d{4}",
    r"\d{2,4}",
]

print("DEMO KESALAHPAHAMAN UMUM")
print("=" * 80)

for text in texts:
    print(f"\nTEXT: {repr(text)}")
    for p in patterns:
        print(f"{p:12} -> {re.findall(p, text)}")

DEMO KESALAHPAHAMAN UMUM

TEXT: 'a'
[a-z]        -> ['a']
[a-z]+       -> ['a']
[a-z]*       -> ['a', '']
[a-z]?       -> ['a', '']
\d{4}        -> []
\d{2,4}      -> []

TEXT: 'ab'
[a-z]        -> ['a', 'b']
[a-z]+       -> ['ab']
[a-z]*       -> ['ab', '']
[a-z]?       -> ['a', 'b', '']
\d{4}        -> []
\d{2,4}      -> []

TEXT: ''
[a-z]        -> []
[a-z]+       -> []
[a-z]*       -> ['']
[a-z]?       -> ['']
\d{4}        -> []
\d{2,4}      -> []

TEXT: '2026'
[a-z]        -> []
[a-z]+       -> []
[a-z]*       -> ['', '', '', '', '']
[a-z]?       -> ['', '', '', '', '']
\d{4}        -> ['2026']
\d{2,4}      -> ['2026']

TEXT: '12'
[a-z]        -> []
[a-z]+       -> []
[a-z]*       -> ['', '', '']
[a-z]?       -> ['', '', '']
\d{4}        -> []
\d{2,4}      -> ['12']

TEXT: '12345'
[a-z]        -> []
[a-z]+       -> []
[a-z]*       -> ['', '', '', '', '', '']
[a-z]?       -> ['', '', '', '', '', '']
\d{4}        -> ['1234']
\d{2,4}      -> ['1234']


#### 15. Contoh NLP yang Relevan

##### a. Mengambil kata
```python
[A-Za-z]+
```

##### b. Mengambil angka
```python
\d+
```

##### c. Mengambil tahun 4 digit
```python
\d{4}
```

##### d. Menormalkan spasi
```python
\s+
```

##### e. Token dengan apostrof opsional
```python
[A-Za-z]+(?:'[A-Za-z]+)?
```

In [32]:

text = "It's 2026. Pemerintah   sedang   belajar NLP."

print("Kata                 ->", re.findall(r"[A-Za-z]+", text))
print("Angka                ->", re.findall(r"\d+", text))
print("Tahun 4 digit        ->", re.findall(r"\d{4}", text))
print("Token apostrof       ->", re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text))
print("Normalisasi spasi    ->", re.sub(r"\s+", " ", text).strip())

Kata                 -> ['It', 's', 'Pemerintah', 'sedang', 'belajar', 'NLP']
Angka                -> ['2026']
Tahun 4 digit        -> ['2026']
Token apostrof       -> ["It's", 'Pemerintah', 'sedang', 'belajar', 'NLP']
Normalisasi spasi    -> It's 2026. Pemerintah sedang belajar NLP.


#### 16. Ringkasan

- Quantifier menjelaskan **berapa kali** suatu pola boleh muncul.
- `+` berarti satu atau lebih.
- `*` berarti nol atau lebih.
- `?` berarti nol atau satu.
- `{n}` berarti tepat n kali.
- `{m,n}` berarti antara m sampai n kali.
- Quantifier sangat penting dalam tokenization, cleaning, dan ekstraksi pola.

### 4.5 Shortcut Character Classes


#### Apa itu Shortcut Character Classes?

Dalam regex, **shortcut character classes** adalah bentuk singkat untuk himpunan karakter yang sering dipakai.

Daripada menulis pola panjang berulang-ulang, regex menyediakan notasi singkat seperti:

- `\d`
- `\w`
- `\s`

dan versi kebalikannya:

- `\D`
- `\W`
- `\S`

Shortcut ini sangat penting karena sering muncul dalam:
- tokenization
- ekstraksi angka
- cleaning teks
- normalisasi spasi
- parsing log, URL, email, dan teks campuran

Contoh:

- `\d` → digit (`[0-9]`)
- `\D` → bukan digit
- `\w` → word character (huruf, angka, underscore)
- `\W` → bukan word character
- `\s` → whitespace
- `\S` → bukan whitespace

#### Mengapa shortcut ini penting?

Dalam preprocessing NLP, kita sering ingin mengatakan hal-hal seperti:
- ambil semua angka
- ambil semua word characters
- deteksi spasi/tab/newline
- ganti semua whitespace berlebih

Shortcut character classes membuat regex:
- lebih singkat
- lebih mudah dibaca
- lebih mudah diingat
- lebih cepat ditulis

#### 1. `\d` — Digit

Pola:

```python
\d
```

artinya:

> satu karakter digit

Dalam banyak konteks, ini ekuivalen dengan:

```python
[0-9]
```

Contoh cocok:
- `0`
- `5`
- `9`

Contoh tidak cocok:
- `a`
- `_`
- spasi

In [33]:

text = "Tahun 2026 ada 3 semester"

print(r"\d  ->", re.findall(r"\d", text))
print(r"[0-9] ->", re.findall(r"[0-9]", text))

\d  -> ['2', '0', '2', '6', '3']
[0-9] -> ['2', '0', '2', '6', '3']


##### Dengan quantifier

Kalau ingin mengambil angka penuh, biasanya `\d` dikombinasikan dengan quantifier:

```python
\d+
```

artinya:
> satu atau lebih digit

In [34]:

text = "Tahun 2026 ada 3 semester dan 12 kelas"

print(re.findall(r"\d+", text))

['2026', '3', '12']


#### 2. `\D` — Non-Digit

Pola:

```python
\D
```

artinya:

> satu karakter yang **bukan digit**

Ini adalah kebalikan dari `\d`.

In [35]:

text = "NLP2026"

print(r"\d ->", re.findall(r"\d", text))
print(r"\D ->", re.findall(r"\D", text))

\d -> ['2', '0', '2', '6']
\D -> ['N', 'L', 'P']


##### Intuisi
- `\d` mengambil angka
- `\D` mengambil semua yang bukan angka

Ini berguna ketika kita ingin:
- menghapus angka
- memisahkan angka dari teks
- mendeteksi bagian non-numerik

#### 3. `\w` — Word Character

Pola:

```python
\w
```

artinya:

> satu **word character**

Dalam Python regex, `\w` biasanya mencakup:
- huruf
- digit
- underscore `_`

Secara intuitif, `\w` sering dipahami mirip dengan:

```python
[A-Za-z0-9_]
```

Tetapi perilakunya bisa sedikit bergantung pada Unicode / engine regex.

In [36]:

text = "NLP_2026 data-science!"

print(r"\w ->", re.findall(r"\w", text))

\w -> ['N', 'L', 'P', '_', '2', '0', '2', '6', 'd', 'a', 't', 'a', 's', 'c', 'i', 'e', 'n', 'c', 'e']


##### Dengan quantifier

```python
\w+
```

artinya:
> satu atau lebih word characters berurutan

Ini sering dipakai untuk token sederhana.

In [37]:

text = "NLP_2026 data-science!"

print(re.findall(r"\w+", text))

['NLP_2026', 'data', 'science']


##### Analisis
Pada teks:

```text
NLP_2026 data-science!
```

pola `\w+` biasanya menghasilkan:
- `NLP_2026`
- `data`
- `science`

karena:
- underscore termasuk `\w`
- tanda minus `-` tidak termasuk `\w`
- tanda seru `!` tidak termasuk `\w`

#### 4. `\W` — Non-Word Character

Pola:

```python
\W
```

artinya:

> satu karakter yang **bukan** word character

Jadi ini kebalikan dari `\w`.

In [38]:

text = "NLP_2026 data-science!"

print(r"\W ->", re.findall(r"\W", text))

\W -> [' ', '-', '!']


##### Kegunaan
`\W` berguna untuk:
- mendeteksi pemisah antar token
- membersihkan simbol tertentu
- meneliti tanda baca dan delimiter

#### 5. `\s` — Whitespace

Pola:

```python
\s
```

artinya:

> satu karakter whitespace

Whitespaces meliputi:
- spasi biasa
- tab `\t`
- newline `\n`
- beberapa bentuk spasi lain

In [39]:

text = "Ini   spasi\tberlebih\nsekali"

print(re.findall(r"\s", text))
print("Jumlah whitespace:", len(re.findall(r"\s", text)))

[' ', ' ', ' ', '\t', '\n']
Jumlah whitespace: 5


##### Dengan quantifier

```python
\s+
```

artinya:
> satu atau lebih whitespace berurutan

Ini adalah salah satu pola paling penting dalam preprocessing.

In [40]:

text = "Ini   spasi\tberlebih\nsekali"

print("Asli :", repr(text))
print("Normal:", re.sub(r"\s+", " ", text).strip())

Asli : 'Ini   spasi\tberlebih\nsekali'
Normal: Ini spasi berlebih sekali


#### 6. `\S` — Non-Whitespace

Pola:

```python
\S
```

artinya:

> satu karakter yang **bukan whitespace**

Ini kebalikan dari `\s`.

In [41]:

text = "Ini   spasi\tberlebih\nsekali"

print(re.findall(r"\S", text)[:20])
print("Jumlah non-whitespace:", len(re.findall(r"\S", text)))

['I', 'n', 'i', 's', 'p', 'a', 's', 'i', 'b', 'e', 'r', 'l', 'e', 'b', 'i', 'h', 's', 'e', 'k', 'a']
Jumlah non-whitespace: 22


#### 7. Ringkasan Pasangan Shortcut

Ada tiga pasangan utama yang sangat penting:

- `\d` ↔ `\D`
- `\w` ↔ `\W`
- `\s` ↔ `\S`

Artinya:
- huruf kecil berarti kategori utama
- huruf besar berarti kebalikannya

In [42]:

summary_text = "NLP_2026 data science!\nBaris ke-2"

print(r"\d ->", re.findall(r"\d", summary_text))
print(r"\D ->", re.findall(r"\D", summary_text)[:20])

print(r"\w ->", re.findall(r"\w", summary_text)[:20])
print(r"\W ->", re.findall(r"\W", summary_text))

print(r"\s ->", re.findall(r"\s", summary_text))
print(r"\S ->", re.findall(r"\S", summary_text)[:20])

\d -> ['2', '0', '2', '6', '2']
\D -> ['N', 'L', 'P', '_', ' ', 'd', 'a', 't', 'a', ' ', 's', 'c', 'i', 'e', 'n', 'c', 'e', '!', '\n', 'B']
\w -> ['N', 'L', 'P', '_', '2', '0', '2', '6', 'd', 'a', 't', 'a', 's', 'c', 'i', 'e', 'n', 'c', 'e', 'B']
\W -> [' ', ' ', '!', '\n', ' ', '-']
\s -> [' ', ' ', '\n', ' ']
\S -> ['N', 'L', 'P', '_', '2', '0', '2', '6', 'd', 'a', 't', 'a', 's', 'c', 'i', 'e', 'n', 'c', 'e', '!']


#### 8. Perbedaan dengan Character Class Biasa

Shortcut classes sering mirip dengan range biasa, tetapi tidak selalu identik secara konseptual.

Contoh:
- `\d` mirip dengan `[0-9]`
- `\w` mirip dengan `[A-Za-z0-9_]`
- `\s` mirip dengan `[ \t\n\r\f\v]` dalam banyak konteks

Tetapi shortcut lebih ringkas dan lebih umum dipakai.

In [43]:

text = "NLP_2026"

print(r"[A-Za-z0-9_] ->", re.findall(r"[A-Za-z0-9_]", text))
print(r"\w           ->", re.findall(r"\w", text))

[A-Za-z0-9_] -> ['N', 'L', 'P', '_', '2', '0', '2', '6']
\w           -> ['N', 'L', 'P', '_', '2', '0', '2', '6']


##### Catatan penting
Untuk pembelajaran awal:
- `\d` bisa dianggap setara dengan `[0-9]`
- `\w` bisa dianggap mirip dengan `[A-Za-z0-9_]`
- `\s` bisa dianggap mewakili whitespace

Tetapi untuk diskusi yang lebih dalam, perilaku bisa dipengaruhi Unicode dan engine regex.

#### 9. Contoh NLP yang Relevan

Shortcut classes sangat sering dipakai dalam tugas NLP.

In [44]:

text = "Email admin_2026@kampus.ac.id pada 19/04/2026 \n #NLP"

print("Ambil digit:")
print(re.findall(r"\d+", text))

print("\nAmbil token word-like:")
print(re.findall(r"\w+", text))

print("\nAmbil whitespace:")
print(re.findall(r"\s+", text))

print("\nGanti semua whitespace berlebih:")
print(re.sub(r"\s+", " ", text).strip())

Ambil digit:
['2026', '19', '04', '2026']

Ambil token word-like:
['Email', 'admin_2026', 'kampus', 'ac', 'id', 'pada', '19', '04', '2026', 'NLP']

Ambil whitespace:
[' ', ' ', ' ', ' \n ']

Ganti semua whitespace berlebih:
Email admin_2026@kampus.ac.id pada 19/04/2026 #NLP


##### Interpretasi
- `\d+` berguna untuk angka, tahun, ID, atau nomor
- `\w+` berguna untuk tokenisasi sederhana
- `\s+` sangat penting untuk normalisasi spasi

#### 10. Shortcut Classes dalam Tokenization

Regex tokenizer sederhana sering memakai:
- `\w+`
- atau class eksplisit seperti `[A-Za-z0-9_]+`

Contoh:

In [45]:

def tokenizer_w(text: str):
    return re.findall(r"\w+", text)

def tokenizer_explicit(text: str):
    return re.findall(r"[A-Za-z0-9_]+", text)

sample = "NLP_2026 adalah topik-data!"

print("Dengan \\w+        :", tokenizer_w(sample))
print("Dengan explicit    :", tokenizer_explicit(sample))

Dengan \w+        : ['NLP_2026', 'adalah', 'topik', 'data']
Dengan explicit    : ['NLP_2026', 'adalah', 'topik', 'data']


##### Analisis
Pada contoh di atas, `\w+` dan `[A-Za-z0-9_]+` memberi hasil yang mirip untuk teks sederhana.

Tetapi perlu diingat:
- `\w` tidak mencakup tanda minus `-`
- `\w` tidak mencakup tanda baca
- `\w+` bukan tokenizer sempurna untuk semua kasus

#### 11. Shortcut Classes untuk Cleaning

Regex shortcut classes juga sangat berguna dalam cleaning teks.

In [46]:

text = "Baris 1:\tNLP 2026\nBaris 2:   Data Science"

print("Hapus digit:")
print(re.sub(r"\d+", "", text))

print("\nNormalisasi whitespace:")
print(re.sub(r"\s+", " ", text).strip())

print("\nAmbil non-whitespace chunks:")
print(re.findall(r"\S+", text))

Hapus digit:
Baris :	NLP 
Baris :   Data Science

Normalisasi whitespace:
Baris 1: NLP 2026 Baris 2: Data Science

Ambil non-whitespace chunks:
['Baris', '1:', 'NLP', '2026', 'Baris', '2:', 'Data', 'Science']


#### 12. Kesalahan Umum

##### 1. Mengira `\w` berarti “word” dalam arti linguistik penuh
Padahal `\w` hanya berarti **word character**, bukan “kata yang sempurna secara linguistik”.

##### 2. Mengira `\w+` adalah tokenizer universal
Padahal `\w+` tidak menangani:
- `COVID-19` sebagai satu token
- `e-commerce` sebagai satu token
- `don't` sebagai satu token, kecuali dimodifikasi

##### 3. Mengira `\s` hanya berarti spasi biasa
Padahal `\s` juga mencakup tab dan newline.

##### 4. Bingung antara huruf kecil dan huruf besar
- `\d` = digit
- `\D` = non-digit
- `\w` = word character
- `\W` = non-word character
- `\s` = whitespace
- `\S` = non-whitespace

#### 13. Perbandingan yang Perlu Diingat

##### `\d` vs `[0-9]`
Untuk level awal, anggap mirip.

##### `\w` vs `[A-Za-z0-9_]`
Untuk level awal, anggap sangat mirip.

##### `\s` vs spasi literal
- `" "` hanya spasi biasa
- `\s` mencakup beberapa jenis whitespace

In [48]:

text = "A B\tC\nD"

print("Spasi literal:", re.findall(r" ", text))
print(r"\s        :", re.findall(r"\s", text))

Spasi literal: [' ']
\s        : [' ', '\t', '\n']


#### 14. Shortcut Classes dan Preprocessing NLP Indonesia

Pada teks Indonesia, shortcut classes sering cukup berguna untuk:
- memisahkan angka dari teks
- normalisasi spasi
- tokenisasi awal yang sederhana

Tetapi tetap ada keterbatasan, misalnya:
- `anak-anak`
- `COVID-19`
- `Rp10.000`
- `12/III/2026`
- `S.Kom.`

In [49]:

indonesian_text = "Anak-anak belajar NLP_2026 di kelas 12/III/2026, harga Rp10.000."

print(r"\w+ ->", re.findall(r"\w+", indonesian_text))
print(r"\d+ ->", re.findall(r"\d+", indonesian_text))
print(r"\W+ ->", re.findall(r"\W+", indonesian_text))

\w+ -> ['Anak', 'anak', 'belajar', 'NLP_2026', 'di', 'kelas', '12', 'III', '2026', 'harga', 'Rp10', '000']
\d+ -> ['2026', '12', '2026', '10', '000']
\W+ -> ['-', ' ', ' ', ' ', ' ', ' ', '/', '/', ', ', ' ', '.', '.']


##### Analisis
Shortcut classes sangat berguna untuk baseline, tetapi tidak cukup untuk semua kasus domain khusus.
Itulah sebabnya regex yang lebih detail atau tokenizer yang lebih kaya sering dibutuhkan.

#### 15. Ringkasan

- `\d` = digit
- `\D` = non-digit
- `\w` = word character
- `\W` = non-word character
- `\s` = whitespace
- `\S` = non-whitespace

Shortcut classes:
- membuat regex lebih ringkas
- sangat berguna dalam preprocessing
- perlu dipahami batasannya dalam tokenization nyata

### 4.6 Anchors

#### Apa itu Anchor?

Dalam regex, **anchor** adalah simbol yang menyatakan **posisi** di dalam string, bukan karakter yang benar-benar diambil.

#### Inti konsep
- Character class seperti `[A-Z]` mencocokkan **karakter**
- Quantifier seperti `+` menjelaskan **berapa kali pola muncul**
- Anchor seperti `^`, `$`, dan `\b` menjelaskan **di mana pola itu harus berada**

Jadi anchor tidak berarti “huruf apa”, tetapi “posisi apa”.

#### Mengapa Anchors penting?

Dalam NLP, preprocessing, dan pencarian pola, kita sering ingin menyatakan hal-hal seperti:
- kata harus muncul di awal string
- pola harus berada di akhir string
- kata harus cocok sebagai kata utuh, bukan bagian dari kata lain
- baris log harus diawali level tertentu seperti `ERROR`
- file atau URL harus berakhiran format tertentu

#### 1. Anchor `^` — Awal String

Pola:

```python
^data
```

artinya:

> cocok jika string dimulai dengan `data`

`^` tidak cocok dengan karakter tertentu, tetapi dengan **posisi awal** string.

In [50]:

samples = ["data science", "big data", "database", "metadata"]

print("ANCHOR ^")
print("=" * 80)
for s in samples:
    print(f"{s!r:20} ->", bool(re.search(r"^data", s)))

ANCHOR ^
'data science'       -> True
'big data'           -> False
'database'           -> True
'metadata'           -> False


##### Interpretasi
- `"data science"` → cocok, karena dimulai dengan `data`
- `"big data"` → tidak cocok, karena `data` bukan di awal
- `"database"` → cocok, karena string memang mulai dengan `data`
- `"metadata"` → tidak cocok, karena string mulai dengan `meta`

#### 2. Anchor `$` — Akhir String

Pola:

```python
science$
```

artinya:

> cocok jika string diakhiri dengan `science`

Seperti `^`, anchor `$` juga menyatakan posisi, yaitu **akhir string**.

In [51]:

samples = ["data science", "science data", "applied science", "sciences"]

print("ANCHOR $")
print("=" * 80)
for s in samples:
    print(f"{s!r:20} ->", bool(re.search(r"science$", s)))

ANCHOR $
'data science'       -> True
'science data'       -> False
'applied science'    -> True
'sciences'           -> False


#### 3. Menggabungkan `^` dan `$`

Kalau `^` menyatakan awal string dan `$` menyatakan akhir string, maka menggabungkannya berarti kita ingin mencocokkan **seluruh string**.

Contoh:

```python
^data$
```

artinya:

> string harus persis `data`, tidak boleh ada karakter lain sebelum atau sesudahnya

In [52]:

samples = ["data", "data science", "big data", "metadata"]

print("GABUNGAN ^ ... $")
print("=" * 80)
for s in samples:
    print(f"{s!r:20} ->", bool(re.search(r"^data$", s)))

GABUNGAN ^ ... $
'data'               -> True
'data science'       -> False
'big data'           -> False
'metadata'           -> False


#### 4. Anchor `\b` — Word Boundary

Pola:

```python
\bdata\b
```

artinya:

> cocok dengan `data` sebagai **kata utuh**, bukan sebagai bagian dari kata lain

Tanpa word boundary, regex `data` akan cocok juga pada:
- `database`
- `metadata`
- `bigdata`

In [53]:

samples = ["data", "database", "metadata", "big data", "data-driven"]

print(r"ANCHOR \b")
print("=" * 80)
for s in samples:
    print(f"{s!r:20} ->", re.findall(r"\bdata\b", s))

ANCHOR \b
'data'               -> ['data']
'database'           -> []
'metadata'           -> []
'big data'           -> ['data']
'data-driven'        -> ['data']


#### Apa sebenarnya `\b` itu?

`\b` berarti **perbatasan** antara:
- word character (`\w`)
dan
- non-word character (`\W`)

atau batas awal/akhir string.

Jadi `\b` bukan “spasi”, tetapi **transisi** antara area kata dan area non-kata.

In [54]:

text = "data, data! (data) database data-driven"

print(re.findall(r"\bdata\b", text))

['data', 'data', 'data', 'data']


#### 5. Anchor `\B` — Non-Word Boundary

`\B` adalah kebalikan dari `\b`.

Pola:

```python
\Bdata\B
```

artinya:

> `data` harus berada **di dalam** kata, bukan di tepi batas kata

In [55]:

samples = ["data", "database", "metadata", "big data", "bigdataanalysis"]

print(r"ANCHOR \B")
print("=" * 80)
for s in samples:
    print(f"{s!r:20} ->", re.findall(r"\Bdata\B", s))

ANCHOR \B
'data'               -> []
'database'           -> []
'metadata'           -> []
'big data'           -> []
'bigdataanalysis'    -> ['data']


#### 6. Perbedaan “mengandung” vs “diawali” vs “diakhiri” vs “kata utuh”

In [56]:

samples = ["data", "database", "big data", "metadata science", "science data"]

patterns = {
    "data": r"data",
    "^data": r"^data",
    "data$": r"data$",
    r"\bdata\b": r"\bdata\b",
}

print("PERBANDINGAN POLA")
print("=" * 80)
for s in samples:
    print(f"\nTEXT: {s!r}")
    for name, pattern in patterns.items():
        print(f"{name:12} ->", bool(re.search(pattern, s)))

PERBANDINGAN POLA

TEXT: 'data'
data         -> True
^data        -> True
data$        -> True
\bdata\b     -> True

TEXT: 'database'
data         -> True
^data        -> True
data$        -> False
\bdata\b     -> False

TEXT: 'big data'
data         -> True
^data        -> False
data$        -> True
\bdata\b     -> True

TEXT: 'metadata science'
data         -> True
^data        -> False
data$        -> False
\bdata\b     -> False

TEXT: 'science data'
data         -> True
^data        -> False
data$        -> True
\bdata\b     -> True


##### Ringkasan perbedaan
- `data` → cocok jika mengandung `data` di mana saja
- `^data` → cocok jika dimulai dengan `data`
- `data$` → cocok jika diakhiri dengan `data`
- `\bdata\b` → cocok jika `data` muncul sebagai kata utuh

#### 7. Anchors pada Multi-Line Text

Pada teks multi-baris, anchor bisa punya perilaku khusus jika memakai flag `re.MULTILINE`.

Tanpa `re.MULTILINE`:
- `^` hanya cocok di awal seluruh string
- `$` hanya cocok di akhir seluruh string

Dengan `re.MULTILINE`:
- `^` cocok di awal setiap baris
- `$` cocok di akhir setiap baris

In [57]:

text = "INFO start\nERROR failed\nINFO retry"

print("Tanpa MULTILINE:")
print(re.findall(r"^ERROR.*$", text))

print("\nDengan MULTILINE:")
print(re.findall(r"^ERROR.*$", text, flags=re.MULTILINE))

Tanpa MULTILINE:
[]

Dengan MULTILINE:
['ERROR failed']


#### 8. Anchors untuk Log Analysis

In [58]:

logs = "INFO start\nERROR database down\nWARN retrying\nERROR timeout\nINFO done"

print(re.findall(r"^ERROR.*$", logs, flags=re.MULTILINE))

['ERROR database down', 'ERROR timeout']


#### 9. Anchors untuk Validasi Format

Anchor `^` dan `$` sangat penting untuk validasi format.

Contoh:
- tahun 4 digit saja
- kode tertentu
- string yang harus sepenuhnya cocok dengan pola

In [59]:

samples = ["2026", "26", "2026a", "a2026"]

print("VALIDASI TAHUN 4 DIGIT")
print("=" * 80)
for s in samples:
    print(f"{s!r:10} ->", bool(re.search(r"^\d{4}$", s)))

VALIDASI TAHUN 4 DIGIT
'2026'     -> True
'26'       -> False
'2026a'    -> False
'a2026'    -> False


##### Interpretasi
`^\d{4}$` berarti:
- `^` awal string
- `\d{4}` tepat empat digit
- `$` akhir string

Jadi string harus benar-benar hanya terdiri dari empat digit.

#### 10. Word Boundary dalam NLP

`\b` sangat penting saat kita ingin mencari kata utuh.

Contoh NLP:
- mencari kata `data` tanpa menangkap `database`
- mendeteksi kata `hoax` sebagai term utuh
- mencari `AI` sebagai token, bukan bagian dari kata yang lebih panjang

In [60]:

text = "AI is different from said, aid, and rail. AI matters."

print("Tanpa boundary :", re.findall(r"AI", text))
print("Dengan boundary:", re.findall(r"\bAI\b", text))

Tanpa boundary : ['AI', 'AI']
Dengan boundary: ['AI', 'AI']


#### 11. Anchor dan Tanda Baca

Karena `\b` berbasis batas antara word dan non-word character, maka tanda baca sering membentuk boundary.

Contoh:
- `data,`
- `(data)`
- `data!`

Semua itu biasanya tetap cocok untuk `\bdata\b`.

In [61]:

text = "data, (data) data! database metadata"

print(re.findall(r"\bdata\b", text))

['data', 'data', 'data']


#### 12. Kesalahan Umum Mahasiswa

1. Mengira `^data` berarti “mengandung data”
2. Mengira `data$` berarti “mengandung data di akhir kalimat mana pun”
3. Mengira `\b` sama dengan spasi
4. Mengira `^data$` sama dengan `data`
5. Lupa bahwa `\b` bergantung pada konsep `\w`

#### 13. Perbandingan dengan `re.match()` dan `re.fullmatch()`

Kadang anchor bisa dibandingkan dengan fungsi regex tertentu:

- `re.match(pattern, text)` mencocokkan dari awal string
- `re.fullmatch(pattern, text)` mencocokkan seluruh string

In [62]:

print(bool(re.match(r"data", "data science")))
print(bool(re.search(r"^data", "data science")))

print(bool(re.fullmatch(r"data", "data")))
print(bool(re.search(r"^data$", "data")))

True
True
True
True


#### 14. Kapan Anchors sangat berguna?

Anchors sangat berguna ketika kita ingin:
- validasi format string
- membedakan substring vs kata utuh
- mencari pola di awal atau akhir baris
- mengambil baris log tertentu
- melakukan filtering teks yang lebih presisi

In [63]:

examples = [
    ("^INFO", "INFO start"),
    ("^INFO", "ERROR INFO start"),
    (r"\bdata\b", "big data"),
    (r"\bdata\b", "database"),
    (r"^\d{4}$", "2026"),
    (r"^\d{4}$", "year 2026"),
]

print("CONTOH PRAKTIS ANCHORS")
print("=" * 80)

for pattern, text in examples:
    print(f"pattern={pattern!r:12} text={text!r:20} ->", bool(re.search(pattern, text)))

CONTOH PRAKTIS ANCHORS
pattern='^INFO'      text='INFO start'         -> True
pattern='^INFO'      text='ERROR INFO start'   -> False
pattern='\\bdata\\b' text='big data'           -> True
pattern='\\bdata\\b' text='database'           -> False
pattern='^\\d{4}$'   text='2026'               -> True
pattern='^\\d{4}$'   text='year 2026'          -> False


#### 15. Ringkasan

- Anchor menyatakan **posisi**, bukan karakter
- `^` = awal string
- `$` = akhir string
- `\b` = word boundary
- `\B` = non-word boundary
- `^...$` sangat berguna untuk validasi format
- `\b...\b` sangat berguna untuk pencarian kata utuh
- Dalam multi-line text, `re.MULTILINE` mengubah perilaku `^` dan `$`

### 4.7 Grouping: `()`

#### Apa itu Grouping?

Dalam regex, **grouping** berarti mengelompokkan sebagian pola agar diperlakukan sebagai satu unit.

Ini biasanya dilakukan dengan tanda kurung:

```python
(...)
```

### Mengapa grouping penting?
Grouping berguna untuk:
- menyatukan beberapa bagian pola
- menerapkan quantifier pada satu blok pola
- menangkap bagian tertentu dari hasil match
- menyusun pola yang lebih kompleks dan lebih terstruktur

##### Intuisi sederhana

Misalkan kita punya pola:

```python
abc+
```

Pola ini berarti:
- `a`
- lalu `b`
- lalu `c+` → huruf `c` satu kali atau lebih

Jadi quantifier `+` hanya berlaku untuk `c`.

Kalau kita ingin seluruh blok `abc` diulang, kita perlu grouping:

```python
(abc)+
```

Sekarang `+` berlaku untuk seluruh grup `abc`.

In [64]:

samples = ["abc", "abcc", "abcabc", "abcabcabc"]

print("TANPA GROUPING: abc+")
print("=" * 80)
for s in samples:
    print(f"{s!r:15} ->", re.findall(r"abc+", s))

print("\nDENGAN GROUPING: (abc)+")
print("=" * 80)
for s in samples:
    print(f"{s!r:15} ->", re.findall(r"(abc)+", s))

TANPA GROUPING: abc+
'abc'           -> ['abc']
'abcc'          -> ['abcc']
'abcabc'        -> ['abc', 'abc']
'abcabcabc'     -> ['abc', 'abc', 'abc']

DENGAN GROUPING: (abc)+
'abc'           -> ['abc']
'abcc'          -> ['abc']
'abcabc'        -> ['abc']
'abcabcabc'     -> ['abc']


##### Penjelasan
- `abc+` berarti `ab` diikuti `c` satu kali atau lebih
- `(abc)+` berarti blok `abc` diulang satu kali atau lebih

Ini menunjukkan bahwa grouping mengubah **cakupan** quantifier.

#### 1. Capturing Group: `()`

Bentuk paling dasar grouping adalah:

```python
(...)
```

Ini disebut **capturing group** karena selain mengelompokkan pola, regex juga “menangkap” isi yang cocok di dalam grup tersebut.

In [65]:

text = "Tanggal: 19/04/2026"

match = re.search(r"(\d{2})/(\d{2})/(\d{4})", text)

print("Match penuh :", match.group(0))
print("Group 1     :", match.group(1))
print("Group 2     :", match.group(2))
print("Group 3     :", match.group(3))

Match penuh : 19/04/2026
Group 1     : 19
Group 2     : 04
Group 3     : 2026


##### Interpretasi
Pada pola:

```python
(\d{2})/(\d{2})/(\d{4})
```

- grup 1 menangkap hari
- grup 2 menangkap bulan
- grup 3 menangkap tahun

Grouping di sini tidak hanya menyusun pola, tetapi juga memecah hasil menjadi bagian-bagian yang dapat diakses.

##### 2. Group 0 vs Group 1, 2, 3, ...

Ini konsep penting:
- `group(0)` = seluruh match
- `group(1)` = isi capturing group pertama
- `group(2)` = isi capturing group kedua
- dan seterusnya

In [66]:

text = "Nomor: 0812-3456-7890"
match = re.search(r"(\d{4})-(\d{4})-(\d{4})", text)

print("group(0):", match.group(0))
print("group(1):", match.group(1))
print("group(2):", match.group(2))
print("group(3):", match.group(3))

group(0): 0812-3456-7890
group(1): 0812
group(2): 3456
group(3): 7890


##### 3. Grouping untuk Menerapkan Quantifier pada Satu Blok

Salah satu fungsi terpenting grouping adalah membuat quantifier berlaku untuk satu blok pola.

Contoh:
- `ha+` berarti `h` lalu `a` satu atau lebih kali
- `(ha)+` berarti blok `ha` diulang satu atau lebih kali

In [67]:

samples = ["ha", "haa", "hahaha", "hahaaa"]

print("ha+")
print("=" * 80)
for s in samples:
    print(f"{s!r:12} ->", re.findall(r"ha+", s))

print("\n(ha)+")
print("=" * 80)
for s in samples:
    print(f"{s!r:12} ->", re.findall(r"(ha)+", s))

ha+
'ha'         -> ['ha']
'haa'        -> ['haa']
'hahaha'     -> ['ha', 'ha', 'ha']
'hahaaa'     -> ['ha', 'haaa']

(ha)+
'ha'         -> ['ha']
'haa'        -> ['ha']
'hahaha'     -> ['ha']
'hahaaa'     -> ['ha']


##### Analisis
- `ha+` menangkap `h` diikuti `a` berulang
- `(ha)+` menangkap pengulangan blok `ha`

Ini penting karena secara struktur keduanya berbeda.

#### 4. Nested Grouping

Grouping dapat disusun di dalam grouping lain. Ini disebut **nested groups**.

Contoh:

```python
((ab)(cd))
```

Di sini ada:
- satu grup besar
- dua grup kecil di dalamnya

In [68]:

text = "abcd"
match = re.search(r"((ab)(cd))", text)

print("group(0):", match.group(0))
print("group(1):", match.group(1))
print("group(2):", match.group(2))
print("group(3):", match.group(3))

group(0): abcd
group(1): abcd
group(2): ab
group(3): cd


##### Interpretasi
- `group(0)` = seluruh match
- `group(1)` = `abcd`
- `group(2)` = `ab`
- `group(3)` = `cd`

Urutan numbering group mengikuti urutan tanda kurung buka dari kiri ke kanan.

#### 5. Grouping dan Alternation

Grouping sering dipakai bersama operator alternation `|`.

Contoh:

```python
(cat|dog)
```

artinya:
> cocok dengan `cat` atau `dog`

Tanpa grouping, alternation bisa memberi hasil yang berbeda dari yang diinginkan.

In [69]:

text = "I have a cat and a dog"

print(re.findall(r"(cat|dog)", text))

['cat', 'dog']


##### Mengapa grouping penting di sini?
Karena kita ingin alternatif berlaku pada satu unit pilihan:
- `cat`
- atau `dog`

Grouping membantu membuat pola lebih eksplisit dan lebih mudah dibaca.

#### 6. Non-Capturing Group: `(?:...)`

Kadang kita ingin mengelompokkan pola, tetapi **tidak ingin menangkap** isinya sebagai grup terpisah.

Untuk itu digunakan:

```python
(?:...)
```

Ini disebut **non-capturing group**.

In [70]:

text = "hahaha"

print("Capturing    :", re.findall(r"(ha)+", text))
print("Non-capturing:", re.findall(r"(?:ha)+", text))

Capturing    : ['ha']
Non-capturing: ['hahaha']


##### Mengapa hasilnya bisa berbeda?

Pada `findall()`:
- jika pola mengandung capturing group, yang dikembalikan sering kali isi grup
- jika pola memakai non-capturing group, yang dikembalikan adalah match penuh

Ini salah satu hal yang paling sering membingungkan mahasiswa.

#### 7. Efek Capturing Group pada `findall()`

Perhatikan contoh ini.

In [71]:

text = "Tanggal 19/04/2026 dan 20/05/2027"

print("Tanpa group:")
print(re.findall(r"\d{2}/\d{2}/\d{4}", text))

print("\nDengan capturing groups:")
print(re.findall(r"(\d{2})/(\d{2})/(\d{4})", text))

Tanpa group:
['19/04/2026', '20/05/2027']

Dengan capturing groups:
[('19', '04', '2026'), ('20', '05', '2027')]


##### Analisis
- Tanpa group, `findall()` mengembalikan full matches
- Dengan capturing groups, `findall()` mengembalikan tuple isi grup

Jadi grouping memengaruhi **bentuk output**, bukan hanya pola.

#### 8. Kapan memakai Capturing vs Non-Capturing Group?

##### Pakai capturing group `()` jika:
- Anda ingin mengambil bagian tertentu dari match
- Anda ingin akses `group(1)`, `group(2)`, dst.
- Anda memang ingin output terstruktur

##### Pakai non-capturing group `(?:...)` jika:
- Anda hanya ingin mengelompokkan pola
- Anda tidak butuh isi grup secara terpisah
- Anda ingin `findall()` mengembalikan full match, bukan isi grup

In [72]:

text = "I don't know if it's right or dont"

pattern_capture = r"[A-Za-z]+('[A-Za-z]+)?"
pattern_noncapture = r"[A-Za-z]+(?:'[A-Za-z]+)?"

print("Capturing:")
print(re.findall(pattern_capture, text))

print("\nNon-capturing:")
print(re.findall(pattern_noncapture, text))

Capturing:
['', "'t", '', '', "'s", '', '', '']

Non-capturing:
['I', "don't", 'know', 'if', "it's", 'right', 'or', 'dont']


##### Penjelasan
Pada pola apostrof opsional:
- capturing group membuat `findall()` fokus ke bagian apostrof
- non-capturing group menjaga agar hasil tetap berupa token penuh

#### 9. Grouping untuk Format Tertentu

Grouping sangat berguna untuk pola terstruktur seperti:
- tanggal
- nomor telepon
- kode tertentu
- pasangan key-value

In [73]:

samples = [
    "Tanggal: 19/04/2026",
    "No HP: 0812-3456-7890",
    "Kode: ABC-123",
]

date_match = re.search(r"(\d{2})/(\d{2})/(\d{4})", samples[0])
phone_match = re.search(r"(\d{4})-(\d{4})-(\d{4})", samples[1])
code_match = re.search(r"([A-Z]{3})-(\d{3})", samples[2])

print("Tanggal:", date_match.groups())
print("Phone  :", phone_match.groups())
print("Kode   :", code_match.groups())

Tanggal: ('19', '04', '2026')
Phone  : ('0812', '3456', '7890')
Kode   : ('ABC', '123')


#### 10. Grouping dalam NLP

Dalam NLP, grouping sering dipakai untuk:
- menangkap bagian tertentu dari pola teks
- membangun tokenizer atau extractor yang lebih fleksibel
- menangani bagian opsional
- menyusun pola dengan struktur yang jelas

In [74]:

text = "I don't think it's impossible, but dont worry."

pattern = r"[A-Za-z]+(?:'[A-Za-z]+)?"
print(re.findall(pattern, text))

['I', "don't", 'think', "it's", 'impossible', 'but', 'dont', 'worry']


##### Analisis
Regex di atas memakai grouping untuk menangani token dengan apostrof seperti:
- `don't`
- `it's`

Jika grouping tidak digunakan dengan tepat, token bisa terpecah atau output `findall()` bisa tidak sesuai harapan.

#### 11. Named Group (Pengantar)

Selain grup biasa, Python regex juga mendukung **named groups**:

```python
(?P<name>...)
```

Ini berguna agar grup tidak hanya diakses dengan nomor, tetapi juga nama.

In [75]:

text = "Tanggal: 19/04/2026"
match = re.search(r"(?P<hari>\d{2})/(?P<bulan>\d{2})/(?P<tahun>\d{4})", text)

print(match.group("hari"))
print(match.group("bulan"))
print(match.group("tahun"))
print(match.groupdict())

19
04
2026
{'hari': '19', 'bulan': '04', 'tahun': '2026'}


##### Mengapa named group berguna?
Karena pada pola besar, mengingat `group(7)` atau `group(8)` bisa membingungkan.  
Nama seperti `hari`, `bulan`, `tahun` jauh lebih mudah dipahami.

#### 12. Grouping vs Character Class

Ini penting: grouping `()` berbeda dengan character class `[]`.

- `()` = mengelompokkan **pola**
- `[]` = menyatakan pilihan **karakter tunggal**

Contoh:
- `(ab)` berarti blok string `ab`
- `[ab]` berarti satu karakter: `a` atau `b`

In [76]:

text = "ab a b"

print("(ab)  ->", re.findall(r"(ab)", text))
print("[ab]  ->", re.findall(r"[ab]", text))

(ab)  -> ['ab']
[ab]  -> ['a', 'b', 'a', 'b']


##### Interpretasi
- `(ab)` cocok dengan substring `ab`
- `[ab]` cocok dengan karakter `a` atau `b` satu per satu

Ini perbedaan mendasar dan sangat sering membingungkan pemula.

#### 13. Kesalahan Umum Mahasiswa

##### 1. Mengira `()` sama dengan `[]`
Padahal fungsinya sangat berbeda.

##### 2. Tidak sadar bahwa quantifier hanya berlaku pada elemen tepat sebelumnya
Akibatnya lupa memakai grouping saat ingin mengulang satu blok.

##### 3. Bingung kenapa `findall()` berubah output-nya ketika ada capturing group
Ini karena capturing group memang memengaruhi hasil.

##### 4. Memakai capturing group padahal hanya butuh pengelompokan struktur
Dalam kasus seperti ini, non-capturing group lebih tepat.

##### 5. Lupa urutan numbering group
Numbering group mengikuti urutan tanda kurung buka dari kiri ke kanan.

In [77]:

tests = [
    ("abcabc", r"(abc)+"),
    ("hahaha", r"(ha)+"),
    ("19/04/2026", r"(\d{2})/(\d{2})/(\d{4})"),
    ("don't its it's", r"[A-Za-z]+('[A-Za-z]+)?"),
    ("don't its it's", r"[A-Za-z]+(?:'[A-Za-z]+)?"),
]

print("DEMO KESALAHPAHAMAN UMUM")
print("=" * 80)

for text, pattern in tests:
    print(f"\nTEXT   : {text!r}")
    print(f"PATTERN: {pattern}")
    print("RESULT :", re.findall(pattern, text))

DEMO KESALAHPAHAMAN UMUM

TEXT   : 'abcabc'
PATTERN: (abc)+
RESULT : ['abc']

TEXT   : 'hahaha'
PATTERN: (ha)+
RESULT : ['ha']

TEXT   : '19/04/2026'
PATTERN: (\d{2})/(\d{2})/(\d{4})
RESULT : [('19', '04', '2026')]

TEXT   : "don't its it's"
PATTERN: [A-Za-z]+('[A-Za-z]+)?
RESULT : ["'t", '', "'s"]

TEXT   : "don't its it's"
PATTERN: [A-Za-z]+(?:'[A-Za-z]+)?
RESULT : ["don't", 'its', "it's"]


### 4.9 Alternation: `|`

#### Apa itu Alternation?

Dalam regex, **alternation** ditulis dengan operator:

```python
|
```

dan secara umum berarti:

> **atau**

Jadi pola:

```python
cat|dog
```

artinya:

> cocok dengan `cat` **atau** `dog`

Alternation sangat penting ketika kita ingin membuat regex yang dapat mencocokkan **beberapa kemungkinan pola**.

#### Mengapa Alternation penting?

Dalam preprocessing NLP dan pencarian pola, kita sering ingin menyatakan:

- cocok dengan satu kata **atau** kata lain
- cocok dengan satu format **atau** format lain
- cocok dengan beberapa level log seperti `INFO`, `WARN`, atau `ERROR`
- cocok dengan beberapa bentuk ejaan
- cocok dengan beberapa domain istilah

Tanpa alternation, kita harus menulis banyak regex terpisah.

#### 1. Alternation Dasar

Pola:

```python
cat|dog
```

artinya:

> cocok dengan `cat` atau `dog`

In [78]:

text = "I have a cat, a dog, and a bird."

print(re.findall(r"cat|dog", text))

['cat', 'dog']


##### Interpretasi
Regex di atas hanya mengambil:
- `cat`
- `dog`

karena hanya dua itu yang didefinisikan sebagai alternatif.

#### 2. Alternation dengan Lebih dari Dua Pilihan

Alternation dapat dipakai untuk banyak kemungkinan:

```python
cat|dog|bird
```

artinya:

> cocok dengan `cat`, `dog`, atau `bird`

In [79]:

text = "I have a cat, a dog, and a bird."

print(re.findall(r"cat|dog|bird", text))

['cat', 'dog', 'bird']


#### 3. Alternation Bekerja pada Pola, Bukan Hanya Kata

Alternation tidak hanya untuk kata literal. Ia bisa dipakai untuk hampir semua pola regex.

Contoh:
- `\d{4}|\d{2}` → empat digit atau dua digit
- `https|http` → `https` atau `http`
- `NLP|AI|ML` → salah satu singkatan tertentu

In [80]:

text = "Tahun 2026, kelas 12, dan kode 99"

print(re.findall(r"\d{4}|\d{2}", text))

['2026', '12', '99']


##### Catatan
Alternation dapat menghubungkan:
- literal dengan literal
- literal dengan pola
- pola dengan pola

#### 4. Alternation dan Prioritas Pembacaan

Ini sangat penting.

Pola:

```python
abc|ab
```

akan mencoba alternatif dari kiri ke kanan.

Jika dua alternatif sama-sama mungkin cocok, engine biasanya akan mengambil **alternatif pertama yang berhasil**.

In [81]:

text = "abc ab"

print(re.findall(r"abc|ab", text))
print(re.findall(r"ab|abc", text))

['abc', 'ab']
['ab', 'ab']


##### Interpretasi
Urutan alternatif itu penting.

- `abc|ab` cenderung mencoba `abc` dulu
- `ab|abc` cenderung mencoba `ab` dulu

Karena itu, susunan alternatif dapat memengaruhi hasil.

#### 5. Alternation dan Grouping

Alternation sering perlu digabung dengan grouping agar maknanya tepat.

Contoh:

```python
(cat|dog)s?
```

artinya:

> `cat` atau `dog`, dengan `s` opsional di belakang

Tanpa grouping, arti regex bisa berubah.

In [82]:

text = "cat cats dog dogs bird birds"

print(re.findall(r"(cat|dog)s?", text))
print(re.findall(r"(?:cat|dog)s?", text))

['cat', 'cat', 'dog', 'dog']
['cat', 'cats', 'dog', 'dogs']


##### Analisis
- `(cat|dog)s?` memakai grouping agar pilihan `cat` atau `dog` dianggap satu unit
- `s?` lalu berlaku pada hasil pilihan itu

Jika memakai capturing group, `findall()` sering mengembalikan isi grup, bukan full match.  
Karena itu, non-capturing group `(?:...)` sering lebih nyaman bila kita hanya ingin struktur.

In [83]:

text = "cat cats dog dogs"

print("Capturing    :", re.findall(r"(cat|dog)s?", text))
print("Non-capturing:", re.findall(r"(?:cat|dog)s?", text))

Capturing    : ['cat', 'cat', 'dog', 'dog']
Non-capturing: ['cat', 'cats', 'dog', 'dogs']


#### 6. Mengapa Grouping Penting dalam Alternation?

Bandingkan dua pola berikut:

```python
ab|cd
```

dan

```python
a(b|c)d
```

Keduanya berbeda.

In [84]:

texts = ["ab", "cd", "abd", "acd", "ad"]

print("Pola: ab|cd")
for t in texts:
    print(f"{t!r:5} ->", bool(re.search(r"ab|cd", t)))

print("\nPola: a(b|c)d")
for t in texts:
    print(f"{t!r:5} ->", bool(re.search(r"a(b|c)d", t)))

Pola: ab|cd
'ab'  -> True
'cd'  -> True
'abd' -> True
'acd' -> True
'ad'  -> False

Pola: a(b|c)d
'ab'  -> False
'cd'  -> False
'abd' -> True
'acd' -> True
'ad'  -> False


##### Interpretasi
- `ab|cd` berarti cocok dengan `ab` atau `cd`
- `a(b|c)d` berarti cocok dengan:
  - `abd`
  - atau `acd`

Grouping mengubah **cakupan** alternation.

#### 7. Alternation untuk Variasi Ejaan

Alternation sering dipakai untuk bentuk ejaan yang berbeda.

Contoh:
- `color|colour`
- `analyze|analyse`
- `organization|organisation`

In [85]:

text = "color colour analyze analyse organization organisation"

print(re.findall(r"color|colour", text))
print(re.findall(r"analyze|analyse", text))
print(re.findall(r"organization|organisation", text))

['color', 'colour']
['analyze', 'analyse']
['organization', 'organisation']


##### Alternatif yang lebih ringkas
Kadang alternation bisa dipadukan dengan grouping atau quantifier opsional agar regex lebih pendek.

Contoh:
- `colou?r`
- `organi[sz]ation`

Tetapi untuk pengantar, alternation eksplisit sering lebih mudah dipahami dulu.

In [86]:

text = "color colour"

print("Alternation eksplisit:", re.findall(r"color|colour", text))
print("Bentuk lebih ringkas :", re.findall(r"colou?r", text))

Alternation eksplisit: ['color', 'colour']
Bentuk lebih ringkas : ['color', 'colour']


#### 8. Alternation untuk Log Analysis

Dalam log, kita sering ingin mengambil beberapa level log sekaligus.

Contoh:
- `INFO|WARN|ERROR`

In [87]:

logs = "INFO start\nERROR failed\nWARN retrying\nDEBUG detail\nINFO done"

print(re.findall(r"INFO|WARN|ERROR", logs))

['INFO', 'ERROR', 'WARN', 'INFO']


##### Pengembangan
Alternation bisa digabung dengan anchor agar lebih presisi, misalnya:

```python
^(INFO|WARN|ERROR)
```

untuk mencocokkan baris yang diawali level log tertentu.

In [88]:

logs = "INFO start\nERROR failed\nWARN retrying\nDEBUG detail\nINFO done"

print(re.findall(r"^(?:INFO|WARN|ERROR).*$", logs, flags=re.MULTILINE))

['INFO start', 'ERROR failed', 'WARN retrying', 'INFO done']


#### 9. Alternation untuk Istilah NLP

Dalam text mining, kita mungkin ingin mencari sejumlah istilah tertentu sekaligus.

Contoh:
- `NLP|AI|ML`
- `hoax|disinformasi|misinformasi`
- `positif|negatif|netral`

In [89]:

text = "Topik kuliah hari ini meliputi NLP, AI, dan ML."

print(re.findall(r"NLP|AI|ML", text))

['NLP', 'AI', 'ML']


In [90]:

text = "Sentimen dapat diklasifikasikan menjadi positif, negatif, atau netral."

print(re.findall(r"positif|negatif|netral", text))

['positif', 'negatif', 'netral']


#### 10. Alternation dan Word Boundary

Kalau kita ingin alternatif dicocokkan sebagai **kata utuh**, sebaiknya alternation dibungkus dengan boundary.

Contoh:

```python
\b(cat|dog)\b
```

artinya:
> `cat` atau `dog`, tetapi harus sebagai kata utuh

In [91]:

text = "cat scatter dog dogmatic cat"

print("Tanpa boundary :", re.findall(r"cat|dog", text))
print("Dengan boundary:", re.findall(r"\b(?:cat|dog)\b", text))

Tanpa boundary : ['cat', 'cat', 'dog', 'dog', 'cat']
Dengan boundary: ['cat', 'dog', 'cat']


##### Analisis
Tanpa `\b`, regex bisa menangkap substring di dalam kata lain.  
Dengan `\b`, hasil menjadi lebih presisi untuk pencarian term utuh.

#### 11. Alternation pada Bagian Opsional

Alternation juga bisa dipakai untuk pola yang lebih kompleks.

Contoh:

```python
Mr|Mrs|Ms
```

atau versi terstruktur:

```python
(?:Mr|Mrs|Ms)\.?
```

yang berarti:
- `Mr`
- `Mrs`
- `Ms`
- titik di belakang opsional

In [92]:

text = "Mr Smith, Mrs Johnson, Ms Lee, Mr. Brown"

print(re.findall(r"(?:Mr|Mrs|Ms)\.?", text))

['Mr', 'Mr', 'Ms', 'Mr.']


In [93]:

text = "cat dog bird"

print("Tanpa group:")
print(re.findall(r"cat|dog", text))

print("\nDengan capturing group:")
print(re.findall(r"(cat|dog)", text))

print("\nDengan non-capturing group:")
print(re.findall(r"(?:cat|dog)", text))

Tanpa group:
['cat', 'dog']

Dengan capturing group:
['cat', 'dog']

Dengan non-capturing group:
['cat', 'dog']


##### Catatan
Pada contoh sederhana ini hasilnya tampak sama, tetapi dalam pola yang lebih kompleks, capturing group bisa mengubah bentuk output `findall()`.

#### 13. Alternation vs Character Class

Ini sangat penting.

- `(cat|dog)` berarti **substring** `cat` atau `dog`
- `[cd]` berarti **satu karakter** `c` atau `d`

Jadi alternation `|` sangat berbeda dari character class `[]`.

In [94]:

text = "cat dog"

print("(cat|dog) ->", re.findall(r"(cat|dog)", text))
print("[cd]      ->", re.findall(r"[cd]", text))

(cat|dog) -> ['cat', 'dog']
[cd]      -> ['c', 'd']


##### Interpretasi
- `(cat|dog)` cocok dengan unit kata `cat` atau `dog`
- `[cd]` cocok dengan karakter tunggal `c` atau `d`

Ini salah satu perbedaan paling penting dalam regex.

#### 14. Alternation dan Greedy/Matching Order

Alternation memilih alternatif berdasarkan urutan dan kecocokan dari kiri ke kanan.

Contoh:
- `data|database`
- `database|data`

Urutannya bisa memengaruhi hasil pada sebagian konteks.

In [95]:

text = "database data"

print(re.findall(r"data|database", text))
print(re.findall(r"database|data", text))

['data', 'data']
['database', 'data']


##### Saran praktis
Kalau ada alternatif yang lebih panjang dan lebih spesifik, sering kali lebih aman menaruhnya lebih dulu.

Misalnya:
- `database|data`
lebih aman daripada:
- `data|database`

#### 15. Kesalahan Umum Mahasiswa

##### 1. Mengira `|` berarti “dan”
Padahal `|` berarti **atau**.

##### 2. Lupa bahwa urutan alternatif bisa memengaruhi hasil
Alternatif dibaca dari kiri ke kanan.

##### 3. Tidak memakai grouping saat alternation harus berlaku pada satu blok
Ini bisa membuat regex bermakna berbeda dari yang dimaksud.

##### 4. Mengira `(cat|dog)` sama dengan `[catdog]`
Padahal:
- `(cat|dog)` = kata `cat` atau `dog`
- `[catdog]` = satu karakter dari himpunan `c,a,t,d,o,g`

##### 5. Tidak sadar bahwa capturing group dalam alternation bisa mengubah output `findall()`

In [96]:

tests = [
    ("cat cats dog dogs", r"(cat|dog)s?"),
    ("cat cats dog dogs", r"(?:cat|dog)s?"),
    ("database data", r"data|database"),
    ("database data", r"database|data"),
]

print("DEMO KESALAHPAHAMAN UMUM")
print("=" * 80)

for text, pattern in tests:
    print(f"\nTEXT   : {text!r}")
    print(f"PATTERN: {pattern}")
    print("RESULT :", re.findall(pattern, text))

DEMO KESALAHPAHAMAN UMUM

TEXT   : 'cat cats dog dogs'
PATTERN: (cat|dog)s?
RESULT : ['cat', 'cat', 'dog', 'dog']

TEXT   : 'cat cats dog dogs'
PATTERN: (?:cat|dog)s?
RESULT : ['cat', 'cats', 'dog', 'dogs']

TEXT   : 'database data'
PATTERN: data|database
RESULT : ['data', 'data']

TEXT   : 'database data'
PATTERN: database|data
RESULT : ['database', 'data']


#### 16. Ringkasan

- Alternation `|` berarti **atau**
- Alternation dipakai untuk menyatakan beberapa kemungkinan pola
- Alternation dapat menghubungkan literal maupun pola regex
- Grouping sangat penting agar cakupan alternation tepat
- Alternation sering dipakai untuk:
  - variasi ejaan
  - istilah tertentu
  - level log
  - beberapa format yang berbeda
- Urutan alternatif bisa memengaruhi hasil

### 4.10 Escaping

#### Apa itu Escaping?

Dalam regex, beberapa karakter memiliki **makna khusus**.  
Contohnya:

- `.`
- `*`
- `+`
- `?`
- `|`
- `(`
- `)`
- `[`
- `]`
- `{`
- `}`
- `^`
- `$`
- `\`

Kalau kita ingin mencocokkan karakter-karakter itu **sebagai karakter biasa**, bukan sebagai operator regex, maka kita harus melakukan **escaping**.

Escaping biasanya dilakukan dengan menambahkan backslash `\` di depan karakter tersebut.

Contoh:
- `\.` berarti titik literal
- `\+` berarti tanda plus literal
- `\?` berarti tanda tanya literal

#### Mengapa Escaping penting?

Tanpa escaping, regex bisa berarti sesuatu yang sama sekali berbeda dari yang kita maksud.

Contoh:

```python
3.10
```

Dalam regex, titik `.` berarti:
> karakter apa saja

Jadi pola `3.10` tidak hanya cocok dengan `3.10`, tetapi juga bisa cocok dengan:
- `3a10`
- `3-10`
- `3X10`

Kalau kita ingin titik literal, kita harus menulis:

```python
3\.10
```

#### 1. Karakter Regex yang Sering Harus Di-escape

Karakter yang sering punya makna khusus dalam regex adalah:

- `.` titik
- `*` bintang
- `+` plus
- `?` tanda tanya
- `|` alternation
- `(` `)` grouping
- `[` `]` character class
- `{` `}` quantifier
- `^` anchor awal
- `$` anchor akhir
- `\` backslash

Jika ingin mencocokkan bentuk literalnya, biasanya perlu escape.

#### 2. Titik `.`

Dalam regex:

```python
.
```

berarti:

> satu karakter apa saja

Jadi titik **bukan** titik literal, kecuali di-escape.

In [97]:

text = "Versi 3.10, 3a10, dan 3-10"

print("Tanpa escape titik:")
print(re.findall(r"3.10", text))

print("\nDengan escape titik:")
print(re.findall(r"3\.10", text))

Tanpa escape titik:
['3.10', '3a10', '3-10']

Dengan escape titik:
['3.10']


#### 3. Tanda Plus `+`

Dalam regex:

```python
+
```

berarti quantifier:
> satu kali atau lebih

Kalau ingin mencocokkan tanda plus literal, gunakan:

```python
\+
```

In [98]:

text = "Nilai A+ lebih tinggi dari A dan B+"

print("Tanpa escape +:")
try:
    print(re.findall(r"A+", text))
except Exception as e:
    print("Error:", e)

print("\nDengan escape +:")
print(re.findall(r"A\+", text))

Tanpa escape +:
['A', 'A']

Dengan escape +:
['A+']


##### Analisis
- `A+` berarti huruf `A` satu kali atau lebih
- `A\+` berarti string literal `A+`

#### 4. Tanda Tanya `?`

Dalam regex:

```python
?
```

berarti:
> nol atau satu kali

Kalau ingin mencocokkan tanda tanya literal, gunakan:

```python
\?
```

In [99]:

text = "Apa? Benarkah? Atau tidak?"

print(re.findall(r"\?", text))

['?', '?', '?']


#### 5. Tanda Bintang `*`

Dalam regex:

```python
*
```

berarti:
> nol kali atau lebih

Kalau ingin mencari karakter bintang literal, gunakan:

```python
\*
```

In [100]:

text = "Wildcard * sering dipakai. Ada juga ** di markdown."

print(re.findall(r"\*", text))

['*', '*', '*']


#### 6. Tanda Kurung `()` dan `[]`

- `()` dipakai untuk grouping
- `[]` dipakai untuk character class

Kalau ingin mencocokkan kurung literal, harus di-escape:

- `\(` dan `\)`
- `\[` dan `\]`

In [101]:

text = "Fungsi f(x) memakai [x] sebagai simbol."

print("Kurung biasa:")
print(re.findall(r"\(.*?\)", text))

print("\nBracket literal:")
print(re.findall(r"\[.*?\]", text))

Kurung biasa:
['(x)']

Bracket literal:
['[x]']


#### 7. Kurawal `{}`

Dalam regex, kurawal dipakai untuk quantifier:

- `{3}`
- `{2,4}`

Kalau ingin mencocokkan kurawal literal, gunakan:

- `\{`
- `\}`

In [102]:

text = "Set dalam matematika bisa ditulis {a, b, c}"

print(re.findall(r"\{.*?\}", text))

['{a, b, c}']


#### 8. Tanda Pipa `|`

Dalam regex, `|` berarti **alternation**, yaitu “atau”.

Kalau ingin mencocokkan simbol pipa literal, gunakan:

```python
\|
```

In [103]:

text = "Pilihan bisa ditulis A|B|C"

print(re.findall(r"\|", text))

['|', '|']


#### 9. Caret `^` dan Dollar `$`

Di banyak posisi:
- `^` berarti awal string
- `$` berarti akhir string

Kalau ingin mencocokkan simbol literalnya, gunakan:
- `\^`
- `\$`

In [104]:

text = "Harga $100 naik ^ tajam"

print("Dollar literal:", re.findall(r"\$", text))
print("Caret literal :", re.findall(r"\^", text))

Dollar literal: ['$']
Caret literal : ['^']


#### 10. Backslash `\`

Backslash sendiri adalah karakter yang sangat khusus karena dipakai untuk escaping.

Kalau ingin mencocokkan backslash literal, kita perlu menulis:

```python
\\
```

Tetapi di Python, backslash juga dipakai untuk escape string.  
Karena itu, ini menjadi topik yang sangat penting.

In [105]:

text = r"Folder C:\Users\Nama\Documents"

print(re.findall(r"\\", text))

['\\', '\\', '\\']


#### 11. Escaping di Regex vs Escaping di Python

Ini bagian yang paling sering membingungkan mahasiswa.

Ada **dua level** yang perlu dipahami:

1. **Python string parsing**
2. **Regex parsing**

Artinya, sebelum regex dibaca oleh engine regex, string-nya lebih dulu dibaca oleh Python.

Karena itu, penulisan backslash bisa terasa “dobel”.

##### Contoh tanpa raw string

Kalau menulis:

```python
"\\d+"
```

Python membaca `\\` sebagai satu backslash literal, sehingga regex yang diterima engine adalah:

```python
\d+
```

##### Contoh dengan raw string

Kalau menulis:

```python
r"\d+"
```

hasil akhirnya juga regex:

```python
\d+
```

Tetapi penulisannya jauh lebih nyaman.

Karena itu, dalam praktik regex Python, **raw string sangat disarankan**.

In [106]:

patterns = [
    "\\d+",
    r"\d+"
]

for p in patterns:
    print(p)

\d+
\d+


#### 12. Mengapa Raw String Sangat Disarankan?

Karena banyak pola regex memakai backslash:
- `\d`
- `\w`
- `\s`
- `\b`
- `\.`

Kalau tidak memakai raw string, penulisan jadi lebih rumit:
- `"\\d+"`
- `"\\w+"`
- `"\\."`

Dengan raw string:
- `r"\d+"`
- `r"\w+"`
- `r"\."`

lebih ringkas dan lebih mudah dibaca.

#### 13. Kapan Karakter Tidak Perlu Di-escape?

Tidak semua karakter harus di-escape.

Contoh huruf dan angka biasa:
- `a`
- `b`
- `1`
- `9`

umumnya bisa ditulis langsung.

Jadi escaping hanya dipakai ketika:
- karakter punya makna khusus di regex
- atau kita ingin aman/eksplisit pada konteks tertentu

In [107]:

text = "data science 2026"

print(re.findall(r"data", text))
print(re.findall(r"2026", text))

['data']
['2026']


#### 14. Escaping di Dalam Character Class `[]`

Di dalam `[]`, beberapa aturan berubah.

Contoh:
- titik `.` di dalam `[]` biasanya tidak perlu di-escape
- tanda plus `+` di dalam `[]` biasanya juga tidak spesial
- tetapi `]`, `-`, `^`, dan `\` bisa perlu perhatian khusus

Contoh:
- `[.]` berarti titik literal
- `[+]` berarti plus literal

In [108]:

text = "Versi 3.10 + 4.20"

print("Titik dengan class:", re.findall(r"[.]", text))
print("Plus dengan class :", re.findall(r"[+]", text))

Titik dengan class: ['.', '.']
Plus dengan class : ['+']


##### Tentang minus `-`
Di dalam character class, `-` sering dipakai untuk range.

Contoh:
- `[a-z]`

Kalau ingin mencocokkan minus literal, biasanya:
- letakkan di awal: `[-abc]`
- atau di akhir: `[abc-]`
- atau escape: `[a\-z]`

In [109]:

text = "COVID-19 e-commerce anak-anak"

print(re.findall(r"[-]", text))
print(re.findall(r"[A-Za-z-]+", text))

['-', '-', '-']
['COVID-', 'e-commerce', 'anak-anak']


#### 15. Escaping untuk URL, Domain, dan Versi

Escaping sering diperlukan untuk pola nyata seperti:
- domain dengan titik
- versi software
- alamat file
- simbol mata uang

In [110]:

text = "Buka https://example.com atau lihat versi 3.10"

print("Domain .com literal:")
print(re.findall(r"\.com", text))

print("\nVersi 3.10 literal:")
print(re.findall(r"3\.10", text))

Domain .com literal:
['.com']

Versi 3.10 literal:
['3.10']


#### 16. Escaping untuk Tokenization dan Preprocessing NLP

Dalam NLP, escaping sering dipakai ketika:
- ingin mempertahankan simbol tertentu
- ingin mencocokkan pola literal secara presisi
- ingin membersihkan karakter spesifik
- ingin mengekstrak struktur tertentu dari teks

In [111]:

text = "Email: admin@example.com, versi 3.10, harga $100, pertanyaan?"

print("Titik literal :", re.findall(r"\.", text))
print("At symbol     :", re.findall(r"@", text))
print("Dollar literal:", re.findall(r"\$", text))
print("Question mark :", re.findall(r"\?", text))

Titik literal : ['.', '.']
At symbol     : ['@']
Dollar literal: ['$']
Question mark : ['?']


#### 17. Contoh Perbandingan Salah vs Benar

##### Kasus 1: Titik literal
- salah: `3.10`
- benar: `3\.10`

##### Kasus 2: Tanda plus literal
- salah: `A+`
- benar: `A\+`

##### Kasus 3: Tanda tanya literal
- salah: `?`
- benar: `\?`

##### Kasus 4: Tanda kurung literal
- salah: `(test)`
- benar: `\(test\)` jika memang ingin kurung literal

In [112]:

text = "A+ B+ versi 3.10 (test)?"

print("Salah untuk A+  :", re.findall(r"A+", text))
print("Benar untuk A+  :", re.findall(r"A\+", text))

print("Salah untuk 3.10:", re.findall(r"3.10", text))
print("Benar untuk 3.10:", re.findall(r"3\.10", text))

print("Kurung literal  :", re.findall(r"\(test\)", text))
print("Tanya literal   :", re.findall(r"\?", text))

Salah untuk A+  : ['A']
Benar untuk A+  : ['A+']
Salah untuk 3.10: ['3.10']
Benar untuk 3.10: ['3.10']
Kurung literal  : ['(test)']
Tanya literal   : ['?']


#### 18. Kesalahan Umum

##### 1. Lupa bahwa `.` bukan titik literal
Akibatnya regex terlalu longgar.

##### 2. Lupa bahwa `+`, `*`, `?` punya makna quantifier
Akibatnya pola tidak cocok seperti yang diharapkan.

##### 3. Bingung antara escape di Python dan escape di regex
Inilah sebab raw string sangat dianjurkan.

##### 4. Meng-escape terlalu banyak atau terlalu sedikit
Tidak semua karakter harus di-escape, tetapi karakter operator utama harus dipahami.

##### 5. Mengira `\b` selalu berarti backspace
Dalam regex, `\b` adalah word boundary, tetapi dalam string Python biasa bisa membingungkan jika tidak memakai raw string.

In [113]:

examples = [
    ("3.10", r"3.10"),
    ("3.10", r"3\.10"),
    ("A+", r"A+"),
    ("A+", r"A\+"),
    ("(test)", r"\(test\)"),
]

print("DEMO KESALAHPAHAMAN UMUM")
print("=" * 80)

for text, pattern in examples:
    print(f"text={text!r:10} pattern={pattern!r:12} ->", re.findall(pattern, text))

DEMO KESALAHPAHAMAN UMUM
text='3.10'     pattern='3.10'       -> ['3.10']
text='3.10'     pattern='3\\.10'     -> ['3.10']
text='A+'       pattern='A+'         -> ['A']
text='A+'       pattern='A\\+'       -> ['A+']
text='(test)'   pattern='\\(test\\)' -> ['(test)']


#### 19. Ringkasan

- Escaping dipakai ketika kita ingin mencocokkan karakter spesial regex sebagai karakter biasa
- Karakter seperti `.`, `+`, `*`, `?`, `|`, `(`, `)`, `[`, `]`, `{`, `}`, `^`, `$`, `\` sering perlu di-escape
- Raw string `r"..."` sangat disarankan di Python untuk regex
- Escaping penting agar regex lebih presisi dan tidak salah makna
- Dalam NLP, escaping sangat relevan untuk cleaning, tokenization, validasi pola, dan ekstraksi literal

## 5. Fungsi Penting di Modul `re`

Lima fungsi yang sangat sering dipakai:

- `re.findall()` → ambil semua kecocokan
- `re.search()` → cari satu kecocokan pertama
- `re.match()` → cocokkan dari awal string
- `re.sub()` → ganti pola
- `re.split()` → pecah string berdasarkan pola

In [ ]:

text = "Email: dosen@example.com dan admin@kampus.ac.id"

print("findall:", re.findall(r"[\w.-]+@[\w.-]+", text))
print("search :", re.search(r"[\w.-]+@[\w.-]+", text).group())
print("match  :", re.match(r"Email", text).group())
print("sub    :", re.sub(r"[\w.-]+@[\w.-]+", "<EMAIL>", text))
print("split  :", re.split(r"\s+", text))

## 6. Contoh Regex untuk Tugas NLP Umum
Di bawah ini beberapa pola yang sering dipakai dalam preprocessing.

In [ ]:

examples = {
    "angka": r"\d+",
    "kata": r"[A-Za-z]+",
    "huruf_angka": r"[A-Za-z0-9]+",
    "email": r"[\w.-]+@[\w.-]+",
    "url_sederhana": r"https?://\S+|www\.\S+",
    "mention": r"@\w+",
    "hashtag": r"#\w+",
}

text = "Hubungi admin@kampus.ac.id atau buka https://kampus.ac.id #NLP @kelasA 2026"

for name, pattern in examples.items():
    print(name, ":", re.findall(pattern, text))

## 7. Regex dan Tokenization

Sekarang kita masuk ke fungsi tokenizer berbasis regex.

Contoh fungsi:

```python
def regex_word_tokenizer(text: str) -> List[str]:
    return re.findall(r"[A-Za-zÀ-ÿ0-9_]+(?:'[A-Za-z]+)?", text)
```

Tokenizer ini memakai `re.findall()` untuk mengambil semua bagian string yang sesuai pola.

In [ ]:

def regex_word_tokenizer(text: str) -> List[str]:
    return re.findall(r"[A-Za-zÀ-ÿ0-9_]+(?:'[A-Za-z]+)?", text)

## 8. Bedah Regex pada `regex_word_tokenizer`

Pola:

```python
r"[A-Za-zÀ-ÿ0-9_]+(?:'[A-Za-z]+)?"
```

Mari kita pecah.

### Bagian 1
```python
[A-Za-zÀ-ÿ0-9_]+
```

Artinya:
- `A-Z` → huruf kapital
- `a-z` → huruf kecil
- `À-ÿ` → beberapa huruf Latin beraksen
- `0-9` → angka
- `_` → underscore
- `+` → satu atau lebih karakter dari himpunan itu

Bagian ini menangkap token seperti:
- `data`
- `NLP`
- `Indonesia`
- `2026`
- `HbA1c`
- `token_1`

### Bagian 2
```python
(?:'[A-Za-z]+)?
```

Artinya:
- `(?:...)` → non-capturing group
- `'` → apostrof literal
- `[A-Za-z]+` → satu atau lebih huruf
- `?` → opsional

Tujuannya agar token seperti:
- `don't`
- `I'm`
- `it's`

tetap dianggap satu token.

In [ ]:

samples = [
    "Saya belajar NLP.",
    "It's a nice day.",
    "Harga Rp10.000 hari ini.",
    "COVID-19 meningkat.",
    "token_1 berhasil diproses.",
    "I'm learning text-mining."
]

for s in samples:
    print("Input :", s)
    print("Token :", regex_word_tokenizer(s))
    print("-" * 80)

## 9. Kelebihan Regex Tokenizer

Regex tokenizer seperti di atas punya beberapa kelebihan:

- lebih baik daripada `.split()` untuk banyak kasus awal
- mudah dijelaskan ke mahasiswa
- transparan dan dapat dikontrol
- cocok untuk prototipe dan pembelajaran dasar
- cukup efektif untuk banyak teks formal sederhana

## 10. Keterbatasan Regex Tokenizer

Regex tokenizer juga punya keterbatasan penting:

- `COVID-19` bisa terpecah menjadi `COVID` dan `19`
- `anak-anak` bisa terpecah menjadi `anak` dan `anak`
- `e-commerce` bisa menjadi `e` dan `commerce`
- `Rp10.000` belum tertangani ideal
- `12/III/2026` belum tertangani ideal
- singkatan seperti `S.Kom.` atau `Ph.D.` bisa terpecah
- tidak dirancang untuk semua bahasa dan sistem tulisan

In [ ]:

hard_cases = [
    "COVID-19 meningkat.",
    "anak-anak bermain.",
    "e-commerce berkembang.",
    "Harga Rp10.000 turun.",
    "No. perkara: 12/III/2026.",
    "Dr. Andi, S.Kom., hadir."
]

title = "KETERBATASAN REGEX TOKENIZER"
print(title)
print("=" * len(title))

for s in hard_cases:
    print("Input :", s)
    print("Token :", regex_word_tokenizer(s))
    print("-" * 80)

## 11. Regex untuk Cleaning / Preprocessing

Regex juga sangat sering dipakai untuk membersihkan teks.

In [ ]:

text = "Hubungi saya di email@example.com atau kunjungi https://contoh.com #NLP @kelasA"

print("Hapus URL     :", re.sub(r"https?://\S+|www\.\S+", " <URL> ", text))
print("Hapus mention :", re.sub(r"@\w+", " <MENTION> ", text))
print("Hapus hashtag :", re.sub(r"#\w+", " <HASHTAG> ", text))
print("Hapus angka   :", re.sub(r"\d+", " ", "Tahun 2026 ada 3 kelas"))
print("Rapikan spasi :", re.sub(r"\s+", " ", "Ini    spasi\tberlebih\nsekali").strip())

## 12. Regex vs Split Biasa

Perbandingan cepat:
- `.split()` hanya memisah berdasarkan delimiter tertentu
- regex bisa mendeskripsikan pola yang jauh lebih fleksibel

Karena itu, regex lebih cocok untuk preprocessing yang sedikit lebih cerdas.

In [ ]:

text = "Pembangunan, pengembangan, dan evaluasi NLP!"

print("split() :", text.split())
print("regex   :", re.findall(r"[A-Za-z]+", text))

## 13. Regex dalam Perspektif NLP

Secara metodologis, regex termasuk pendekatan **rule-based**:
- eksplisit
- cepat
- mudah dijelaskan
- cocok untuk baseline

Tetapi regex bukan solusi sempurna. Untuk banyak kasus yang lebih kompleks, NLP modern juga memakai:
- rule-based tokenizer yang lebih kaya
- statistical tokenizer
- subword tokenizer seperti BPE atau WordPiece

## 14. Ringkasan Konseptual

- Regex adalah bahasa mini untuk pola teks.
- Regex sangat berguna untuk pencarian, ekstraksi, penggantian, dan tokenization.
- Dalam NLP, regex sering dipakai pada preprocessing awal.
- Regex tokenizer lebih fleksibel daripada `.split()`.
- Regex tokenizer tetap punya keterbatasan pada kasus ambigu dan domain khusus.
- Karena itu, regex adalah fondasi yang sangat baik untuk dipahami, tetapi bukan satu-satunya pendekatan.

## 15. Soal Reflektif

1. Mengapa `split()` tidak cukup untuk banyak kasus NLP?
2. Apa perbedaan `[A-Za-z]+` dan `\w+`?
3. Mengapa `r"\d+"` lebih disukai daripada `"\\d+"`?
4. Mengapa tokenizer regex yang baik untuk berita belum tentu baik untuk log sistem?
5. Bagaimana Anda akan memodifikasi tokenizer agar `COVID-19` tetap menjadi satu token?
6. Kapan regex tokenizer sebaiknya dipakai, dan kapan perlu beralih ke pendekatan lain?